# RepairLab: original-model feasibility baseline

Generate one proposed repair for each of **164 external HumanEvalFix problems**. This is
a **single-response baseline without execution feedback**, not an interactive
agent evaluation, training run, or final held-out result. No weights are trained.

Qwen3-0.6B is the first feasibility candidate because this project's predecessor
already ran it on a free GPU. It is not a claim that this is the best coding model.
Near-zero success would tell us to reconsider difficulty or the starting model.

**Kaggle:** import this notebook, keep it private, enable Internet, select one GPU.
**Colab:** upload the notebook, select a GPU runtime. Run cells in order.
Check your live free quota before starting; this notebook does not read it.
The run has 164 prompts, one response per prompt, and at most 1,024 new tokens
per response. It does not use paid inference or require an API key.

Only public task data and a version fingerprint are embedded. There are no
private cases, reference repairs, or grader in this notebook. Model-generated
Python is saved as text and NEVER executed here. Download the output archive
and evaluate it locally with the command at the end.


In [ ]:
import os
import sys
import subprocess
if sys.version_info < (3, 10):
    raise RuntimeError("Use a Python 3.10+ GPU runtime")
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["HF_HUB_DISABLE_IMPLICIT_TOKEN"] = "1"
import torch
if not torch.cuda.is_available():
    raise RuntimeError("Select a GPU runtime before continuing")
print("GPU:", torch.cuda.get_device_name(0))
subprocess.check_call([sys.executable, "-m", "pip", "install", "transformers==5.2.0"])


In [ ]:
import json
TASKS = json.loads('[{"entry_point":"has_close_elements","public_tests_source":"def check(has_close_elements):\\n    assert has_close_elements([1.0, 2.0, 3.0], 0.5) == False\\n    assert has_close_elements([1.0, 2.8, 3.0, 4.0, 5.0, 2.0], 0.3) == True\\ncheck(has_close_elements)\\n","source":"from typing import List\\n\\n\\ndef has_close_elements(numbers: List[float], threshold: float) -> bool:\\n    for idx, elem in enumerate(numbers):\\n        for idx2, elem2 in enumerate(numbers):\\n            if idx != idx2:\\n                distance = elem - elem2\\n                if distance < threshold:\\n                    return True\\n\\n    return False\\n","specification":"Check if in given list of numbers, are any two numbers closer to each other than\\ngiven threshold.\\n>>> has_close_elements([1.0, 2.0, 3.0], 0.5)\\nFalse\\n>>> has_close_elements([1.0, 2.8, 3.0, 4.0, 5.0, 2.0], 0.3)\\nTrue","task_id":"Python/0"},{"entry_point":"separate_paren_groups","public_tests_source":"def check(separate_paren_groups):\\n    assert separate_paren_groups(\'( ) (( )) (( )( ))\') == [\'()\', \'(())\', \'(()())\']\\ncheck(separate_paren_groups)\\n","source":"from typing import List\\n\\n\\ndef separate_paren_groups(paren_string: str) -> List[str]:\\n    result = []\\n    current_string = []\\n    current_depth = 0\\n\\n    for c in paren_string:\\n        if c == \'(\':\\n            current_depth += 1\\n            current_string.append(c)\\n        elif c == \')\':\\n            current_depth -= 1\\n            current_string.append(c)\\n\\n            if current_depth < 0:\\n                result.append(\'\'.join(current_string))\\n                current_string.clear()\\n\\n    return result\\n","specification":"Input to this function is a string containing multiple groups of nested parentheses. Your goal is to\\nseparate those group into separate strings and return the list of those.\\nSeparate groups are balanced (each open brace is properly closed) and not nested within each other\\nIgnore any spaces in the input string.\\n>>> separate_paren_groups(\'( ) (( )) (( )( ))\')\\n[\'()\', \'(())\', \'(()())\']","task_id":"Python/1"},{"entry_point":"truncate_number","public_tests_source":"def check(truncate_number):\\n    assert truncate_number(3.5) == 0.5\\ncheck(truncate_number)\\n","source":"def truncate_number(number: float) -> float:\\n    return number % 1.0 + 1.0\\n","specification":"Given a positive floating point number, it can be decomposed into\\nand integer part (largest integer smaller than given number) and decimals\\n(leftover part always smaller than 1).\\nReturn the decimal part of the number.\\n>>> truncate_number(3.5)\\n0.5","task_id":"Python/2"},{"entry_point":"below_zero","public_tests_source":"def check(below_zero):\\n    assert below_zero([1, 2, 3]) == False\\n    assert below_zero([1, 2, -4, 5]) == True\\ncheck(below_zero)\\n","source":"from typing import List\\n\\n\\ndef below_zero(operations: List[int]) -> bool:\\n    balance = 0\\n\\n    for op in operations:\\n        balance += op\\n        if balance == 0:\\n            return True\\n\\n    return False\\n","specification":"You\'re given a list of deposit and withdrawal operations on a bank account that starts with\\nzero balance. Your task is to detect if at any point the balance of account fallls below zero, and\\nat that point function should return True. Otherwise it should return False.\\n>>> below_zero([1, 2, 3])\\nFalse\\n>>> below_zero([1, 2, -4, 5])\\nTrue","task_id":"Python/3"},{"entry_point":"mean_absolute_deviation","public_tests_source":"def check(mean_absolute_deviation):\\n    assert abs(mean_absolute_deviation([1.0, 2.0, 3.0, 4.0]) - 1.0) < 1e-6\\ncheck(mean_absolute_deviation)\\n","source":"from typing import List\\n\\n\\ndef mean_absolute_deviation(numbers: List[float]) -> float:\\n    mean = sum(numbers) / len(numbers)\\n    return sum(abs(x - mean) for x in numbers) / mean\\n","specification":"For a given list of input numbers, calculate Mean Absolute Deviation\\naround the mean of this dataset.\\nMean Absolute Deviation is the average absolute difference between each\\nelement and a centerpoint (mean in this case):\\nMAD = average | x - x_mean |\\n>>> mean_absolute_deviation([1.0, 2.0, 3.0, 4.0])\\n1.0","task_id":"Python/4"},{"entry_point":"intersperse","public_tests_source":"def check(intersperse):\\n    assert intersperse([], 4) == []\\n    assert intersperse([1,2,3], 4) == [1,4,2,4,3]\\ncheck(intersperse)\\n","source":"from typing import List\\n\\n\\ndef intersperse(numbers: List[int], delimeter: int) -> List[int]:\\n    if not numbers:\\n        return []\\n\\n    result = []\\n\\n    for n in numbers[:-1]:\\n        result.append(n)\\n        result.append(delimeter)\\n\\n    return result\\n","specification":"Insert a number \'delimeter\' between every two consecutive elements of input list `numbers\'\\n>>> intersperse([], 4)\\n[]\\n>>> intersperse([1, 2, 3], 4)\\n[1, 4, 2, 4, 3]","task_id":"Python/5"},{"entry_point":"parse_nested_parens","public_tests_source":"def check(parse_nested_parens):\\n    assert parse_nested_parens(\'(()()) ((())) () ((())()())\') == [2, 3, 1, 3]\\ncheck(parse_nested_parens)\\n","source":"from typing import List\\n\\n\\ndef parse_nested_parens(paren_string: str) -> List[int]:\\n    def parse_paren_group(s):\\n        depth = 0\\n        max_depth = 0\\n        for c in s:\\n            if c == \'(\':\\n                depth += 1\\n                max_depth = max(depth, max_depth)\\n            else:\\n                max_depth -= 1\\n\\n        return max_depth\\n\\n    return [parse_paren_group(x) for x in paren_string.split(\' \') if x]\\n","specification":"Input to this function is a string represented multiple groups for nested parentheses separated by spaces.\\nFor each of the group, output the deepest level of nesting of parentheses.\\nE.g. (()()) has maximum two levels of nesting while ((())) has three.\\n>>> parse_nested_parens(\'(()()) ((())) () ((())()())\')\\n[2, 3, 1, 3]","task_id":"Python/6"},{"entry_point":"filter_by_substring","public_tests_source":"def check(filter_by_substring):\\n    assert filter_by_substring([], \'a\') == []\\n    assert filter_by_substring([\'abc\', \'bacd\', \'cde\', \'array\'], \'a\') == [\'abc\', \'bacd\', \'array\']\\ncheck(filter_by_substring)\\n","source":"from typing import List\\n\\n\\ndef filter_by_substring(strings: List[str], substring: str) -> List[str]:\\n    return [x for x in strings if x in substring]\\n","specification":"Filter an input list of strings only for ones that contain given substring\\n>>> filter_by_substring([], \'a\')\\n[]\\n>>> filter_by_substring([\'abc\', \'bacd\', \'cde\', \'array\'], \'a\')\\n[\'abc\', \'bacd\', \'array\']","task_id":"Python/7"},{"entry_point":"sum_product","public_tests_source":"def check(sum_product):\\n    assert sum_product([]) == (0, 1)\\n    assert sum_product([1, 2,3,4]) == (10, 24)\\ncheck(sum_product)\\n","source":"from typing import List, Tuple\\n\\n\\ndef sum_product(numbers: List[int]) -> Tuple[int, int]:\\n    sum_value = 0\\n    prod_value = 0\\n\\n    for n in numbers:\\n        sum_value += n\\n        prod_value *= n\\n    return sum_value, prod_value\\n","specification":"For a given list of integers, return a tuple consisting of a sum and a product of all the integers in a list.\\nEmpty sum should be equal to 0 and empty product should be equal to 1.\\n>>> sum_product([])\\n(0, 1)\\n>>> sum_product([1, 2, 3, 4])\\n(10, 24)","task_id":"Python/8"},{"entry_point":"rolling_max","public_tests_source":"def check(rolling_max):\\n    assert rolling_max([1, 2, 3, 2, 3, 4, 2]) == [1, 2, 3, 3, 3, 4, 4]\\ncheck(rolling_max)\\n","source":"from typing import List, Tuple\\n\\n\\ndef rolling_max(numbers: List[int]) -> List[int]:\\n    running_max = None\\n    result = []\\n\\n    for n in numbers:\\n        if running_max is None:\\n            running_max = n\\n        else:\\n            running_max = max(numbers)\\n\\n        result.append(running_max)\\n\\n    return result\\n","specification":"From a given list of integers, generate a list of rolling maximum element found until given moment\\nin the sequence.\\n>>> rolling_max([1, 2, 3, 2, 3, 4, 2])\\n[1, 2, 3, 3, 3, 4, 4]","task_id":"Python/9"},{"entry_point":"make_palindrome","public_tests_source":"def check(make_palindrome):\\n    assert make_palindrome(\'\') == \'\'\\n    assert make_palindrome(\'cat\') == \'catac\'\\n    assert make_palindrome(\'cata\') == \'catac\'\\ncheck(make_palindrome)\\n","source":"def is_palindrome(string: str) -> bool:\\n    \\"\\"\\" Test if given string is a palindrome \\"\\"\\"\\n    return string == string[::-1]\\n\\n\\ndef make_palindrome(string: str) -> str:\\n    if not string:\\n        return \'\'\\n\\n    beginning_of_suffix = 0\\n\\n    while not is_palindrome(string):\\n        beginning_of_suffix += 1\\n\\n    return string + string[:beginning_of_suffix][::-1]\\n","specification":"Find the shortest palindrome that begins with a supplied string.\\nAlgorithm idea is simple:\\n- Find the longest postfix of supplied string that is a palindrome.\\n- Append to the end of the string reverse of a string prefix that comes before the palindromic suffix.\\n>>> make_palindrome(\'\')\\n\'\'\\n>>> make_palindrome(\'cat\')\\n\'catac\'\\n>>> make_palindrome(\'cata\')\\n\'catac\'","task_id":"Python/10"},{"entry_point":"string_xor","public_tests_source":"def check(string_xor):\\n    assert string_xor(\'010\', \'110\') == \'100\'\\ncheck(string_xor)\\n","source":"from typing import List\\n\\n\\ndef string_xor(a: str, b: str) -> str:\\n    def xor(i, j):\\n        if i == j:\\n            return \'1\'\\n        else:\\n            return \'0\'\\n\\n    return \'\'.join(xor(x, y) for x, y in zip(a, b))\\n","specification":"Input are two strings a and b consisting only of 1s and 0s.\\nPerform binary XOR on these inputs and return result also as a string.\\n>>> string_xor(\'010\', \'110\')\\n\'100\'","task_id":"Python/11"},{"entry_point":"longest","public_tests_source":"def check(longest):\\n    assert longest([]) == None\\n    assert longest([\'a\', \'b\', \'c\']) == \'a\'\\n    assert longest([\'a\', \'bb\', \'ccc\']) == \'ccc\'\\ncheck(longest)\\n","source":"from typing import List, Optional\\n\\n\\ndef longest(strings: List[str]) -> Optional[str]:\\n    if not strings:\\n        return None\\n\\n    maxlen = max(len(x) for x in strings)\\n    for s in strings:\\n        if len(s) > maxlen:\\n            return s\\n","specification":"Out of list of strings, return the longest one. Return the first one in case of multiple\\nstrings of the same length. Return None in case the input list is empty.\\n>>> longest([])\\n>>> longest([\'a\', \'b\', \'c\'])\\n\'a\'\\n>>> longest([\'a\', \'bb\', \'ccc\'])\\n\'ccc\'","task_id":"Python/12"},{"entry_point":"greatest_common_divisor","public_tests_source":"def check(greatest_common_divisor):\\n    assert greatest_common_divisor(3, 5) == 1\\n    assert greatest_common_divisor(25, 15) == 5\\ncheck(greatest_common_divisor)\\n","source":"def greatest_common_divisor(a: int, b: int) -> int:\\n    while b:\\n        a, b = b, a % b\\n    return b\\n","specification":"Return a greatest common divisor of two integers a and b\\n>>> greatest_common_divisor(3, 5)\\n1\\n>>> greatest_common_divisor(25, 15)\\n5","task_id":"Python/13"},{"entry_point":"all_prefixes","public_tests_source":"def check(all_prefixes):\\n    assert all_prefixes(\'abc\') == [\'a\', \'ab\', \'abc\']\\ncheck(all_prefixes)\\n","source":"from typing import List\\n\\n\\ndef all_prefixes(string: str) -> List[str]:\\n    result = []\\n\\n    for i in range(len(string)-1):\\n        result.append(string[:i+1])\\n    return result\\n","specification":"Return list of all prefixes from shortest to longest of the input string\\n>>> all_prefixes(\'abc\')\\n[\'a\', \'ab\', \'abc\']","task_id":"Python/14"},{"entry_point":"string_sequence","public_tests_source":"def check(string_sequence):\\n    assert string_sequence(0) == \'0\'\\n    assert string_sequence(5) == \'0 1 2 3 4 5\'\\ncheck(string_sequence)\\n","source":"def string_sequence(n: int) -> str:\\n    return \' \'.join([str(x) for x in range(n)])\\n","specification":"Return a string containing space-delimited numbers starting from 0 upto n inclusive.\\n>>> string_sequence(0)\\n\'0\'\\n>>> string_sequence(5)\\n\'0 1 2 3 4 5\'","task_id":"Python/15"},{"entry_point":"count_distinct_characters","public_tests_source":"def check(count_distinct_characters):\\n    assert count_distinct_characters(\'xyzXYZ\') == 3\\n    assert count_distinct_characters(\'Jerry\') == 4\\ncheck(count_distinct_characters)\\n","source":"def count_distinct_characters(string: str) -> int:\\n    return len(set(string))\\n","specification":"Given a string, find out how many distinct characters (regardless of case) does it consist of\\n>>> count_distinct_characters(\'xyzXYZ\')\\n3\\n>>> count_distinct_characters(\'Jerry\')\\n4","task_id":"Python/16"},{"entry_point":"parse_music","public_tests_source":"def check(parse_music):\\n    assert parse_music(\'o o| .| o| o| .| .| .| .| o o\') == [4, 2, 1, 2, 2, 1, 1, 1, 1, 4, 4]\\ncheck(parse_music)\\n","source":"from typing import List\\n\\n\\ndef parse_music(music_string: str) -> List[int]:\\n    note_map = {\'o\': 3, \'o|\': 2, \'.|\': 1}\\n    return [note_map[x] for x in music_string.split(\' \') if x]\\n","specification":"Input to this function is a string representing musical notes in a special ASCII format.\\nYour task is to parse this string and return list of integers corresponding to how many beats does each\\nnot last.\\nHere is a legend:\\n\'o\' - whole note, lasts four beats\\n\'o|\' - half note, lasts two beats\\n\'.|\' - quater note, lasts one beat\\n>>> parse_music(\'o o| .| o| o| .| .| .| .| o o\')\\n[4, 2, 1, 2, 2, 1, 1, 1, 1, 4, 4]","task_id":"Python/17"},{"entry_point":"how_many_times","public_tests_source":"def check(how_many_times):\\n    assert how_many_times(\'\', \'a\') == 0\\n    assert how_many_times(\'aaa\', \'a\') == 3\\n    assert how_many_times(\'aaaa\', \'aa\') == 3\\ncheck(how_many_times)\\n","source":"def how_many_times(string: str, substring: str) -> int:\\n    times = 0\\n\\n    for i in range(len(string) - len(substring)):\\n        if string[i:i+len(substring)] == substring:\\n            times += 1\\n\\n    return times\\n","specification":"Find how many times a given substring can be found in the original string. Count overlaping cases.\\n>>> how_many_times(\'\', \'a\')\\n0\\n>>> how_many_times(\'aaa\', \'a\')\\n3\\n>>> how_many_times(\'aaaa\', \'aa\')\\n3","task_id":"Python/18"},{"entry_point":"sort_numbers","public_tests_source":"def check(sort_numbers):\\n    assert sort_numbers(\'three one five\') == \'one three five\'\\ncheck(sort_numbers)\\n","source":"from typing import List\\n\\n\\ndef sort_numbers(numbers: str) -> str:\\n    value_map = {\\n        \'zero\': 0,\\n        \'one\': 1,\\n        \'two\': 2,\\n        \'three\': 3,\\n        \'four\': 4,\\n        \'five\': 5,\\n        \'six\': 6,\\n        \'seven\': 7,\\n        \'eight\': 8,\\n        \'nine\': 9\\n    }\\n    return \' \'.join([x for x in numbers.split(\' \') if x])\\n","specification":"Input is a space-delimited string of numberals from \'zero\' to \'nine\'.\\nValid choices are \'zero\', \'one\', \'two\', \'three\', \'four\', \'five\', \'six\', \'seven\', \'eight\' and \'nine\'.\\nReturn the string with numbers sorted from smallest to largest\\n>>> sort_numbers(\'three one five\')\\n\'one three five\'","task_id":"Python/19"},{"entry_point":"find_closest_elements","public_tests_source":"def check(find_closest_elements):\\n    assert find_closest_elements([1.0, 2.0, 3.0, 4.0, 5.0, 2.2]) == (2.0, 2.2)\\n    assert find_closest_elements([1.0, 2.0, 3.0, 4.0, 5.0, 2.0]) == (2.0, 2.0)\\ncheck(find_closest_elements)\\n","source":"from typing import List, Tuple\\n\\n\\ndef find_closest_elements(numbers: List[float]) -> Tuple[float, float]:\\n    closest_pair = None\\n    distance = None\\n\\n    for idx, elem in enumerate(numbers):\\n        for idx2, elem2 in enumerate(numbers):\\n            if idx != idx2:\\n                if distance is None:\\n                    distance = abs(elem - elem2)\\n                    closest_pair = tuple(sorted([elem, elem2]))\\n                else:\\n                    new_distance = abs(elem - elem2)\\n                    if new_distance > distance:\\n                        distance = new_distance\\n                        closest_pair = tuple(sorted([elem, elem2]))\\n\\n    return closest_pair\\n","specification":"From a supplied list of numbers (of length at least two) select and return two that are the closest to each\\nother and return them in order (smaller number, larger number).\\n>>> find_closest_elements([1.0, 2.0, 3.0, 4.0, 5.0, 2.2])\\n(2.0, 2.2)\\n>>> find_closest_elements([1.0, 2.0, 3.0, 4.0, 5.0, 2.0])\\n(2.0, 2.0)","task_id":"Python/20"},{"entry_point":"rescale_to_unit","public_tests_source":"def check(rescale_to_unit):\\n    assert rescale_to_unit([1.0, 2.0, 3.0, 4.0, 5.0]) == [0.0, 0.25, 0.5, 0.75, 1.0]\\ncheck(rescale_to_unit)\\n","source":"from typing import List\\n\\n\\ndef rescale_to_unit(numbers: List[float]) -> List[float]:\\n    min_number = min(numbers)\\n    max_number = max(numbers)\\n    return [(x - min_number) / (max_number + min_number) for x in numbers]\\n","specification":"Given list of numbers (of at least two elements), apply a linear transform to that list,\\nsuch that the smallest number will become 0 and the largest will become 1\\n>>> rescale_to_unit([1.0, 2.0, 3.0, 4.0, 5.0])\\n[0.0, 0.25, 0.5, 0.75, 1.0]","task_id":"Python/21"},{"entry_point":"filter_integers","public_tests_source":"def check(filter_integers):\\n    assert filter_integers([\'a\', 3.14, 5]) == [5]\\n    assert filter_integers([1, 2, 3, \'abc\', {}, []]) == [1,2,3]\\ncheck(filter_integers)\\n","source":"from typing import List, Any\\n\\n\\ndef filter_integers(values: List[Any]) -> List[int]:\\n    out = [x for x in values if isinstance(x, int)]\\n    return values\\n","specification":"Filter given list of any python values only for integers\\n>>> filter_integers([\'a\', 3.14, 5])\\n[5]\\n>>> filter_integers([1, 2, 3, \'abc\', {}, []])\\n[1, 2, 3]","task_id":"Python/22"},{"entry_point":"strlen","public_tests_source":"def check(strlen):\\n    assert strlen(\'\') == 0\\n    assert strlen(\'abc\') == 3\\ncheck(strlen)\\n","source":"def strlen(string: str) -> int:\\n    return len(string) - 1\\n","specification":"Return length of given string\\n>>> strlen(\'\')\\n0\\n>>> strlen(\'abc\')\\n3","task_id":"Python/23"},{"entry_point":"largest_divisor","public_tests_source":"def check(largest_divisor):\\n    assert largest_divisor(15) == 5\\ncheck(largest_divisor)\\n","source":"def largest_divisor(n: int) -> int:\\n    for i in reversed(range(n)):\\n        if n - i == 0:\\n            return i\\n","specification":"For a given number n, find the largest number that divides n evenly, smaller than n\\n>>> largest_divisor(15)\\n5","task_id":"Python/24"},{"entry_point":"factorize","public_tests_source":"def check(factorize):\\n    assert factorize(8) == [2, 2, 2]\\n    assert factorize(25) == [5,5]\\n    assert factorize(70) == [2,5,7]\\ncheck(factorize)\\n","source":"from typing import List\\n\\n\\ndef factorize(n: int) -> List[int]:\\n    import math\\n    fact = []\\n    i = 0\\n    while i <= int(math.sqrt(n) + 1):\\n        if n % i == 0:\\n            fact.append(i)\\n            n //= i\\n        else:\\n            i += 1\\n\\n    if n > 1:\\n        fact.append(n)\\n    return fact\\n","specification":"Return list of prime factors of given integer in the order from smallest to largest.\\nEach of the factors should be listed number of times corresponding to how many times it appeares in factorization.\\nInput number should be equal to the product of all factors\\n>>> factorize(8)\\n[2, 2, 2]\\n>>> factorize(25)\\n[5, 5]\\n>>> factorize(70)\\n[2, 5, 7]","task_id":"Python/25"},{"entry_point":"remove_duplicates","public_tests_source":"def check(remove_duplicates):\\n    assert remove_duplicates([1, 2, 3,2, 4]) == [1, 3, 4]\\ncheck(remove_duplicates)\\n","source":"from typing import List\\n\\n\\ndef remove_duplicates(numbers: List[int]) -> List[int]:\\n    import collections\\n    c = collections.Counter(numbers)\\n    return [n for n in numbers if c[n] < 1]\\n","specification":"From a list of integers, remove all elements that occur more than once.\\nKeep order of elements left the same as in the input.\\n>>> remove_duplicates([1, 2, 3, 2, 4])\\n[1, 3, 4]","task_id":"Python/26"},{"entry_point":"flip_case","public_tests_source":"def check(flip_case):\\n    assert flip_case(\'Hello\') == \'hELLO\'\\ncheck(flip_case)\\n","source":"def flip_case(string: str) -> str:\\n    return string.lower()\\n","specification":"For a given string, flip lowercase characters to uppercase and uppercase to lowercase.\\n>>> flip_case(\'Hello\')\\n\'hELLO\'","task_id":"Python/27"},{"entry_point":"concatenate","public_tests_source":"def check(concatenate):\\n    assert concatenate([]) == \'\'\\n    assert concatenate([\'a\', \'b\', \'c\']) == \'abc\'\\ncheck(concatenate)\\n","source":"from typing import List\\n\\n\\ndef concatenate(strings: List[str]) -> str:\\n    return \' \'.join(strings)\\n","specification":"Concatenate list of strings into a single string\\n>>> concatenate([])\\n\'\'\\n>>> concatenate([\'a\', \'b\', \'c\'])\\n\'abc\'","task_id":"Python/28"},{"entry_point":"filter_by_prefix","public_tests_source":"def check(filter_by_prefix):\\n    assert filter_by_prefix([], \'a\') == []\\n    assert filter_by_prefix([\'abc\', \'bcd\', \'cde\', \'array\'], \'a\') == [\'abc\', \'array\']\\ncheck(filter_by_prefix)\\n","source":"from typing import List\\n\\n\\ndef filter_by_prefix(strings: List[str], prefix: str) -> List[str]:\\n    return [x for x in strings if x.endswith(prefix)]\\n","specification":"Filter an input list of strings only for ones that start with a given prefix.\\n>>> filter_by_prefix([], \'a\')\\n[]\\n>>> filter_by_prefix([\'abc\', \'bcd\', \'cde\', \'array\'], \'a\')\\n[\'abc\', \'array\']","task_id":"Python/29"},{"entry_point":"get_positive","public_tests_source":"def check(get_positive):\\n    assert get_positive([-1, 2, -4, 5, 6]) == [2, 5, 6]\\n    assert get_positive([5, 3, -5, 2, -3, 3, 9, 0, 123, 1, -10]) == [5, 3, 2, 3, 9, 123, 1]\\ncheck(get_positive)\\n","source":"def get_positive(l: list):\\n    return [e for e in l if e < 0]\\n","specification":"Return only positive numbers in the list.\\n>>> get_positive([-1, 2, -4, 5, 6])\\n[2, 5, 6]\\n>>> get_positive([5, 3, -5, 2, -3, 3, 9, 0, 123, 1, -10])\\n[5, 3, 2, 3, 9, 123, 1]","task_id":"Python/30"},{"entry_point":"is_prime","public_tests_source":"def check(is_prime):\\n    assert is_prime(6) == False\\n    assert is_prime(101) == True\\n    assert is_prime(11) == True\\n    assert is_prime(13441) == True\\n    assert is_prime(61) == True\\n    assert is_prime(4) == False\\n    assert is_prime(1) == False\\ncheck(is_prime)\\n","source":"def is_prime(n):\\n    if n < 1:\\n        return False\\n    for k in range(1, n - 1):\\n        if n % k == 0:\\n            return False\\n    return True\\n","specification":"Return true if a given number is prime, and false otherwise.\\n>>> is_prime(6)\\nFalse\\n>>> is_prime(101)\\nTrue\\n>>> is_prime(11)\\nTrue\\n>>> is_prime(13441)\\nTrue\\n>>> is_prime(61)\\nTrue\\n>>> is_prime(4)\\nFalse\\n>>> is_prime(1)\\nFalse","task_id":"Python/31"},{"entry_point":"find_zero","public_tests_source":"def check(find_zero):\\n    assert abs(find_zero([1,2])+0.5<1e-4)\\n    assert abs(find_zero([-6,11,-6,1])-1<1e-4)\\ncheck(find_zero)\\n","source":"import math\\n\\n\\ndef poly(xs: list, x: float):\\n    \\"\\"\\"\\n    Evaluates polynomial with coefficients xs at point x.\\n    return xs[0] + xs[1] * x + xs[1] * x^2 + .... xs[n] * x^n\\n    \\"\\"\\"\\n    return sum([coeff * math.pow(x, i) for i, coeff in enumerate(xs)])\\n\\n\\ndef find_zero(xs: list):\\n    begin, end = -1., 1.\\n    while poly(xs, begin) * poly(xs, end) > 0:\\n        begin *= 2.0\\n        end *= 2.0\\n    while begin - end > 1e-10:\\n        center = (begin + end) / 2.0\\n        if poly(xs, center) * poly(xs, begin) > 0:\\n            begin = center\\n        else:\\n            end = center\\n    return begin\\n","specification":"xs are coefficients of a polynomial.\\nfind_zero find x such that poly(x) = 0.\\nfind_zero returns only only zero point, even if there are many.\\nMoreover, find_zero only takes list xs having even number of coefficients\\nand largest non zero coefficient as it guarantees\\na solution.\\n>>> round(find_zero([1, 2]), 2) # f(x) = 1 + 2x\\n-0.5\\n>>> round(find_zero([-6, 11, -6, 1]), 2) # (x - 1) * (x - 2) * (x - 3) = -6 + 11x - 6x^2 + x^3\\n1.0","task_id":"Python/32"},{"entry_point":"sort_third","public_tests_source":"def check(sort_third):\\n    assert sort_third([1, 2, 3]) == [1, 2, 3]\\n    assert sort_third([5, 6, 3, 4, 8, 9, 2]) == [2, 6, 3, 4, 8, 9, 5]\\ncheck(sort_third)\\n","source":"def sort_third(l: list):\\n    l = list(l)\\n    return l\\n","specification":"This function takes a list l and returns a list l\' such that\\nl\' is identical to l in the indicies that are not divisible by three, while its values at the indicies that are divisible by three are equal\\nto the values of the corresponding indicies of l, but sorted.\\n>>> sort_third([1, 2, 3])\\n[1, 2, 3]\\n>>> sort_third([5, 6, 3, 4, 8, 9, 2])\\n[2, 6, 3, 4, 8, 9, 5]","task_id":"Python/33"},{"entry_point":"unique","public_tests_source":"def check(unique):\\n    assert unique([5, 3, 5, 2, 3, 3, 9, 0, 123]) == [0, 2, 3, 5, 9, 123]\\ncheck(unique)\\n","source":"def unique(l: list):\\n    return sorted(l)\\n","specification":"Return sorted unique elements in a list\\n>>> unique([5, 3, 5, 2, 3, 3, 9, 0, 123])\\n[0, 2, 3, 5, 9, 123]","task_id":"Python/34"},{"entry_point":"max_element","public_tests_source":"def check(max_element):\\n    assert max_element([1, 2, 3]) == 3\\n    assert max_element([5, 3, -5, 2, -3, 3, 9, 0, 123, 1, -10]) == 123\\ncheck(max_element)\\n","source":"def max_element(l: list):\\n    m = l[0]\\n    for e in l:\\n        if e < m:\\n            m = e\\n    return m\\n","specification":"Return maximum element in the list.\\n>>> max_element([1, 2, 3])\\n3\\n>>> max_element([5, 3, -5, 2, -3, 3, 9, 0, 123, 1, -10])\\n123","task_id":"Python/35"},{"entry_point":"fizz_buzz","public_tests_source":"def check(fizz_buzz):\\n    assert fizz_buzz(50) == 0\\n    assert fizz_buzz(78) == 2\\n    assert fizz_buzz(79) == 3\\ncheck(fizz_buzz)\\n","source":"def fizz_buzz(n: int):\\n    ns = []\\n    for i in range(n):\\n        if i % 11 == 0 and i % 13 == 0:\\n            ns.append(i)\\n    s = \'\'.join(list(map(str, ns)))\\n    ans = 0\\n    for c in s:\\n        ans += (c == \'7\')\\n    return ans\\n","specification":"Return the number of times the digit 7 appears in integers less than n which are divisible by 11 or 13.\\n>>> fizz_buzz(50)\\n0\\n>>> fizz_buzz(78)\\n2\\n>>> fizz_buzz(79)\\n3","task_id":"Python/36"},{"entry_point":"sort_even","public_tests_source":"def check(sort_even):\\n    assert tuple(sort_even([1, 2, 3])) == tuple([1, 2, 3])\\n    assert tuple(sort_even([5, 6,3,4])) == tuple([3,6,5,4])\\ncheck(sort_even)\\n","source":"def sort_even(l: list):\\n    evens = l[::2]\\n    odds = l[1::2]\\n    odds.sort()\\n    ans = []\\n    for e, o in zip(evens, odds):\\n        ans.extend([e, o])\\n    if len(evens) > len(odds):\\n        ans.append(evens[-1])\\n    return ans\\n","specification":"This function takes a list l and returns a list l\' such that\\nl\' is identical to l in the odd indicies, while its values at the even indicies are equal\\nto the values of the even indicies of l, but sorted.\\n>>> sort_even([1, 2, 3])\\n[1, 2, 3]\\n>>> sort_even([5, 6, 3, 4])\\n[3, 6, 5, 4]","task_id":"Python/37"},{"entry_point":"decode_cyclic","public_tests_source":"","source":"def encode_cyclic(s: str):\\n    \\"\\"\\"\\n    returns encoded string by cycling groups of three characters.\\n    \\"\\"\\"\\n    # split string to groups. Each of length 3.\\n    groups = [s[(3 * i):min((3 * i + 3), len(s))] for i in range((len(s) + 2) // 3)]\\n    # cycle elements in each group. Unless group has fewer elements than 3.\\n    groups = [(group[1:] + group[0]) if len(group) == 3 else group for group in groups]\\n    return \\"\\".join(groups)\\n\\n\\ndef decode_cyclic(s: str):\\n    return encode_cyclic(s)\\n","specification":"takes as input string encoded with encode_cyclic function. Returns decoded string.","task_id":"Python/38"},{"entry_point":"prime_fib","public_tests_source":"def check(prime_fib):\\n    assert prime_fib(1) == 2\\n    assert prime_fib(2) == 3\\n    assert prime_fib(3) == 5\\n    assert prime_fib(4) == 13\\n    assert prime_fib(5) == 89\\ncheck(prime_fib)\\n","source":"def prime_fib(n: int):\\n    import math\\n\\n    def is_prime(p):\\n        if p < 2:\\n            return False\\n        for k in range(2, min(int(math.sqrt(p)), p)):\\n            if p % k == 0:\\n                return False\\n        return True\\n    f = [0, 1]\\n    while True:\\n        f.append(f[-1] + f[-2])\\n        if is_prime(f[-1]):\\n            n -= 1\\n        if n == 0:\\n            return f[-1]\\n","specification":"prime_fib returns n-th number that is a Fibonacci number and it\'s also prime.\\n>>> prime_fib(1)\\n2\\n>>> prime_fib(2)\\n3\\n>>> prime_fib(3)\\n5\\n>>> prime_fib(4)\\n13\\n>>> prime_fib(5)\\n89","task_id":"Python/39"},{"entry_point":"triples_sum_to_zero","public_tests_source":"def check(triples_sum_to_zero):\\n    assert triples_sum_to_zero([1, 3, 5, 0]) == False\\n    assert triples_sum_to_zero([1, 3, -2, 1]) == True\\n    assert triples_sum_to_zero([1, 2, 3, 7]) == False\\n    assert triples_sum_to_zero([2, 4, -5, 3, 9, 7]) == True\\ncheck(triples_sum_to_zero)\\n","source":"def triples_sum_to_zero(l: list):\\n    for i in range(1, len(l)):\\n        for j in range(i + 1, len(l)):\\n            for k in range(j + 1, len(l)):\\n                if l[i] + l[j] + l[k] == 0:\\n                    return True\\n    return False\\n","specification":"triples_sum_to_zero takes a list of integers as an input.\\nit returns True if there are three distinct elements in the list that\\nsum to zero, and False otherwise.\\n>>> triples_sum_to_zero([1, 3, 5, 0])\\nFalse\\n>>> triples_sum_to_zero([1, 3, -2, 1])\\nTrue\\n>>> triples_sum_to_zero([1, 2, 3, 7])\\nFalse\\n>>> triples_sum_to_zero([2, 4, -5, 3, 9, 7])\\nTrue\\n>>> triples_sum_to_zero([1])\\nFalse","task_id":"Python/40"},{"entry_point":"car_race_collision","public_tests_source":"","source":"def car_race_collision(n: int):\\n    return n**3\\n","specification":"Imagine a road that\'s a perfectly straight infinitely long line.\\nn cars are driving left to right;  simultaneously, a different set of n cars\\nare driving right to left.   The two sets of cars start out being very far from\\neach other.  All cars move in the same speed.  Two cars are said to collide\\nwhen a car that\'s moving left to right hits a car that\'s moving right to left.\\nHowever, the cars are infinitely sturdy and strong; as a result, they continue moving\\nin their trajectory as if they did not collide.\\nThis function outputs the number of such collisions.","task_id":"Python/41"},{"entry_point":"incr_list","public_tests_source":"def check(incr_list):\\n    assert incr_list([1, 2, 3]) == [2, 3, 4]\\n    assert incr_list([5, 2, 5, 2, 3, 3, 9, 0, 123]) == [6, 3, 6, 3, 4, 4, 10, 1, 124]\\ncheck(incr_list)\\n","source":"def incr_list(l: list):\\n    return [(e + 2) for e in l]\\n","specification":"Return list with elements incremented by 1.\\n>>> incr_list([1, 2, 3])\\n[2, 3, 4]\\n>>> incr_list([5, 3, 5, 2, 3, 3, 9, 0, 123])\\n[6, 4, 6, 3, 4, 4, 10, 1, 124]","task_id":"Python/42"},{"entry_point":"pairs_sum_to_zero","public_tests_source":"def check(pairs_sum_to_zero):\\n    assert pairs_sum_to_zero([1, 3, 5, 0]) == False\\n    assert pairs_sum_to_zero([1, 3, -2, 1]) == False\\n    assert pairs_sum_to_zero([1, 2, 3, 7]) == False\\n    assert pairs_sum_to_zero([2, 4, -5, 3, 5, 7]) == True\\ncheck(pairs_sum_to_zero)\\n","source":"def pairs_sum_to_zero(l):\\n    for i, l1 in enumerate(l):\\n        for j in range(i, len(l)):\\n            if l1 + l[j] == 0:\\n                return True\\n    return False\\n","specification":"pairs_sum_to_zero takes a list of integers as an input.\\nit returns True if there are two distinct elements in the list that\\nsum to zero, and False otherwise.\\n>>> pairs_sum_to_zero([1, 3, 5, 0])\\nFalse\\n>>> pairs_sum_to_zero([1, 3, -2, 1])\\nFalse\\n>>> pairs_sum_to_zero([1, 2, 3, 7])\\nFalse\\n>>> pairs_sum_to_zero([2, 4, -5, 3, 5, 7])\\nTrue\\n>>> pairs_sum_to_zero([1])\\nFalse","task_id":"Python/43"},{"entry_point":"change_base","public_tests_source":"def check(change_base):\\n    assert change_base(8, 3) == \\"22\\"\\n    assert change_base(8, 2) == \\"1000\\"\\n    assert change_base(7, 2) == \\"111\\"\\ncheck(change_base)\\n","source":"def change_base(x: int, base: int):\\n    ret = \\"\\"\\n    while x > 0:\\n        ret = str(x % base) + ret\\n        x -= base\\n    return ret\\n","specification":"Change numerical base of input number x to base.\\nreturn string representation after the conversion.\\nbase numbers are less than 10.\\n>>> change_base(8, 3)\\n\'22\'\\n>>> change_base(8, 2)\\n\'1000\'\\n>>> change_base(7, 2)\\n\'111\'","task_id":"Python/44"},{"entry_point":"triangle_area","public_tests_source":"def check(triangle_area):\\n    assert triangle_area(5, 3) == 7.5\\ncheck(triangle_area)\\n","source":"def triangle_area(a, h):\\n    return a * h / 0.5\\n","specification":"Given length of a side and high return area for a triangle.\\n>>> triangle_area(5, 3)\\n7.5","task_id":"Python/45"},{"entry_point":"fib4","public_tests_source":"def check(fib4):\\n    assert fib4(5) == 4\\n    assert fib4(6) == 8\\n    assert fib4(7) == 14\\ncheck(fib4)\\n","source":"def fib4(n: int):\\n    results = [0, 0, 2, 0]\\n    if n < 4:\\n        return results[n]\\n\\n    for _ in range(4, n + 1):\\n        results.append(results[-1] + results[-2] + results[-3] + results[-4])\\n        results.pop(0)\\n\\n    return results[-2]\\n","specification":"The Fib4 number sequence is a sequence similar to the Fibbonacci sequnece that\'s defined as follows:\\nfib4(0) -> 0\\nfib4(1) -> 0\\nfib4(2) -> 2\\nfib4(3) -> 0\\nfib4(n) -> fib4(n-1) + fib4(n-2) + fib4(n-3) + fib4(n-4).\\nPlease write a function to efficiently compute the n-th element of the fib4 number sequence.  Do not use recursion.\\n>>> fib4(5)\\n4\\n>>> fib4(6)\\n8\\n>>> fib4(7)\\n14","task_id":"Python/46"},{"entry_point":"median","public_tests_source":"def check(median):\\n    assert median([3, 1, 2, 4, 5]) == 3\\n    assert median([-10, 4, 6, 1000, 10, 20]) == 8.0\\ncheck(median)\\n","source":"def median(l: list):\\n    l = sorted(l)\\n    if len(l) % 2 == 1:\\n        return l[len(l) // 2]\\n    else:\\n        return (l[len(l) - 1 // 2] + l[len(l) // 2]) / 2.0\\n","specification":"Return median of elements in the list l.\\n>>> median([3, 1, 2, 4, 5])\\n3\\n>>> median([-10, 4, 6, 1000, 10, 20])\\n15.0","task_id":"Python/47"},{"entry_point":"is_palindrome","public_tests_source":"def check(is_palindrome):\\n    assert is_palindrome(\'\') == True\\n    assert is_palindrome(\'aba\') == True\\n    assert is_palindrome(\'aaaaa\') == True\\n    assert is_palindrome(\'zbcd\') == False\\ncheck(is_palindrome)\\n","source":"def is_palindrome(text: str):\\n    for i in range(len(text)):\\n        if text[i] != text[len(text) - i]:\\n            return False\\n    return True\\n","specification":"Checks if given string is a palindrome\\n>>> is_palindrome(\'\')\\nTrue\\n>>> is_palindrome(\'aba\')\\nTrue\\n>>> is_palindrome(\'aaaaa\')\\nTrue\\n>>> is_palindrome(\'zbcd\')\\nFalse","task_id":"Python/48"},{"entry_point":"modp","public_tests_source":"def check(modp):\\n    assert modp(3, 5) == 3\\n    assert modp(1101, 101) == 2\\n    assert modp(0, 101) == 1\\n    assert modp(3, 11) == 8\\n    assert modp(100, 101) == 1\\ncheck(modp)\\n","source":"def modp(n: int, p: int):\\n    ret = 0\\n    for i in range(n):\\n        ret = (2 * ret) % p\\n    return ret\\n","specification":"Return 2^n modulo p (be aware of numerics).\\n>>> modp(3, 5)\\n3\\n>>> modp(1101, 101)\\n2\\n>>> modp(0, 101)\\n1\\n>>> modp(3, 11)\\n8\\n>>> modp(100, 101)\\n1","task_id":"Python/49"},{"entry_point":"decode_shift","public_tests_source":"","source":"def encode_shift(s: str):\\n    \\"\\"\\"\\n    returns encoded string by shifting every character by 5 in the alphabet.\\n    \\"\\"\\"\\n    return \\"\\".join([chr(((ord(ch) + 5 - ord(\\"a\\")) % 26) + ord(\\"a\\")) for ch in s])\\n\\n\\ndef decode_shift(s: str):\\n    return \\"\\".join([chr(((ord(ch) - 5 - ord(\\"a\\")) % 26) + ord(ch)) for ch in s])\\n","specification":"takes as input string encoded with encode_shift function. Returns decoded string.","task_id":"Python/50"},{"entry_point":"remove_vowels","public_tests_source":"def check(remove_vowels):\\n    assert remove_vowels(\'\') == \'\'\\n    assert remove_vowels(\\"abcdef\\\\nghijklm\\") == \'bcdf\\\\nghjklm\'\\n    assert remove_vowels(\'abcdef\') == \'bcdf\'\\n    assert remove_vowels(\'aaaaa\') == \'\'\\n    assert remove_vowels(\'aaBAA\') == \'B\'\\n    assert remove_vowels(\'zbcd\') == \'zbcd\'\\ncheck(remove_vowels)\\n","source":"def remove_vowels(text):\\n    return \\"\\".join([s for s in text if s.lower() not in [\\"a\\", \\"e\\", \\"i\\", \\"o\\", \\"u\\", \\"w\\", \\"y\\"]])\\n","specification":"remove_vowels is a function that takes string and returns string without vowels.\\n>>> remove_vowels(\'\')\\n\'\'\\n>>> remove_vowels(\\"abcdef\\\\nghijklm\\")\\n\'bcdf\\\\nghjklm\'\\n>>> remove_vowels(\'abcdef\')\\n\'bcdf\'\\n>>> remove_vowels(\'aaaaa\')\\n\'\'\\n>>> remove_vowels(\'aaBAA\')\\n\'B\'\\n>>> remove_vowels(\'zbcd\')\\n\'zbcd\'","task_id":"Python/51"},{"entry_point":"below_threshold","public_tests_source":"def check(below_threshold):\\n    assert below_threshold([1, 2, 4, 10], 100)\\n    assert not below_threshold([1, 20, 4, 10], 5)\\ncheck(below_threshold)\\n","source":"def below_threshold(l: list, t: int):\\n    for e in l:\\n        if e >= t:\\n            return True\\n    return False\\n","specification":"Return True if all numbers in the list l are below threshold t.\\n>>> below_threshold([1, 2, 4, 10], 100)\\nTrue\\n>>> below_threshold([1, 20, 4, 10], 5)\\nFalse","task_id":"Python/52"},{"entry_point":"add","public_tests_source":"def check(add):\\n    import random\\n    assert add(2, 3) == 5\\n    assert add(5, 7) == 12\\ncheck(add)\\n","source":"def add(x: int, y: int):\\n    return x + y + y + x\\n","specification":"Add two numbers x and y\\n>>> add(2, 3)\\n5\\n>>> add(5, 7)\\n12","task_id":"Python/53"},{"entry_point":"same_chars","public_tests_source":"def check(same_chars):\\n    assert same_chars(\'eabcdzzzz\', \'dddzzzzzzzddeddabc\') == True\\n    assert same_chars(\'abcd\', \'dddddddabc\') == True\\n    assert same_chars(\'dddddddabc\', \'abcd\') == True\\n    assert same_chars(\'eabcd\', \'dddddddabc\') == False\\n    assert same_chars(\'abcd\', \'dddddddabcf\') == False\\n    assert same_chars(\'eabcdzzzz\', \'dddzzzzzzzddddabc\') == False\\ncheck(same_chars)\\n","source":"def same_chars(s0: str, s1: str):\\n    return s0 == s1\\n","specification":"Check if two words have the same characters.\\n>>> same_chars(\'eabcdzzzz\', \'dddzzzzzzzddeddabc\')\\nTrue\\n>>> same_chars(\'abcd\', \'dddddddabc\')\\nTrue\\n>>> same_chars(\'dddddddabc\', \'abcd\')\\nTrue\\n>>> same_chars(\'eabcd\', \'dddddddabc\')\\nFalse\\n>>> same_chars(\'abcd\', \'dddddddabce\')\\nFalse\\n>>> same_chars(\'eabcdzzzz\', \'dddzzzzzzzddddabc\')\\nFalse","task_id":"Python/54"},{"entry_point":"fib","public_tests_source":"def check(fib):\\n    assert fib(10) == 55\\n    assert fib(1) == 1\\n    assert fib(8) == 21\\ncheck(fib)\\n","source":"def fib(n: int):\\n    if n == 0:\\n        return 0\\n    if n == 1:\\n        return 1\\n    if n == 2:\\n        return 2\\n    return fib(n - 1) + fib(n - 2)\\n","specification":"Return n-th Fibonacci number.\\n>>> fib(10)\\n55\\n>>> fib(1)\\n1\\n>>> fib(8)\\n21","task_id":"Python/55"},{"entry_point":"correct_bracketing","public_tests_source":"def check(correct_bracketing):\\n    assert correct_bracketing(\\"<>\\")\\n    assert correct_bracketing(\\"<<><>>\\")\\n    assert not correct_bracketing(\\"><<>\\")\\n    assert not correct_bracketing(\\"<\\")\\ncheck(correct_bracketing)\\n","source":"def correct_bracketing(brackets: str):\\n    depth = 0\\n    for b in brackets:\\n        if b == \\">\\":\\n            depth += 1\\n        else:\\n            depth -= 1\\n        if depth < 0:\\n            return False\\n    return depth == 0\\n","specification":"brackets is a string of \\"<\\" and \\">\\".\\nreturn True if every opening bracket has a corresponding closing bracket.\\n>>> correct_bracketing(\\"<\\")\\nFalse\\n>>> correct_bracketing(\\"<>\\")\\nTrue\\n>>> correct_bracketing(\\"<<><>>\\")\\nTrue\\n>>> correct_bracketing(\\"><<>\\")\\nFalse","task_id":"Python/56"},{"entry_point":"monotonic","public_tests_source":"def check(monotonic):\\n    assert monotonic([1, 2, 4, 10]) == True\\n    assert monotonic([1, 20, 4, 10]) == False\\n    assert monotonic([4, 1, 0, -10]) == True\\ncheck(monotonic)\\n","source":"def monotonic(l: list):\\n    if l == sorted(l) or l == sorted(l, reverse=True):\\n        return False\\n    return True\\n","specification":"Return True is list elements are monotonically increasing or decreasing.\\n>>> monotonic([1, 2, 4, 20])\\nTrue\\n>>> monotonic([1, 20, 4, 10])\\nFalse\\n>>> monotonic([4, 1, 0, -10])\\nTrue","task_id":"Python/57"},{"entry_point":"common","public_tests_source":"def check(common):\\n    assert common([1, 4, 3, 34, 653, 2, 5], [5, 7, 1, 5, 9, 653, 121]) == [1, 5, 653]\\n    assert common([5, 3, 2, 8], [3, 2]) == [2, 3]\\ncheck(common)\\n","source":"def common(l1: list, l2: list):\\n    ret = set()\\n    for e1 in l1:\\n        for e2 in l2:\\n            ret.add(e1)\\n    return sorted(list(ret))\\n","specification":"Return sorted unique common elements for two lists.\\n>>> common([1, 4, 3, 34, 653, 2, 5], [5, 7, 1, 5, 9, 653, 121])\\n[1, 5, 653]\\n>>> common([5, 3, 2, 8], [3, 2])\\n[2, 3]","task_id":"Python/58"},{"entry_point":"largest_prime_factor","public_tests_source":"def check(largest_prime_factor):\\n    assert largest_prime_factor(2048) == 2\\n    assert largest_prime_factor(13195) == 29\\ncheck(largest_prime_factor)\\n","source":"def largest_prime_factor(n: int):\\n    def is_prime(k):\\n        if k < 2:\\n            return False\\n        for i in range(2, k - 1):\\n            if k % i == 0:\\n                return False\\n        return True\\n    largest = 1\\n    for j in range(2, n + 1):\\n        if n % j == 0 and is_prime(n):\\n            largest = max(largest, j)\\n    return largest\\n","specification":"Return the largest prime factor of n. Assume n > 1 and is not a prime.\\n>>> largest_prime_factor(13195)\\n29\\n>>> largest_prime_factor(2048)\\n2","task_id":"Python/59"},{"entry_point":"sum_to_n","public_tests_source":"def check(sum_to_n):\\n    assert sum_to_n(1) == 1\\n    assert sum_to_n(5) == 15\\n    assert sum_to_n(10) == 55\\n    assert sum_to_n(30) == 465\\n    assert sum_to_n(100) == 5050\\ncheck(sum_to_n)\\n","source":"def sum_to_n(n: int):\\n    return sum(range(n))\\n","specification":"sum_to_n is a function that sums numbers from 1 to n.\\n>>> sum_to_n(30)\\n465\\n>>> sum_to_n(100)\\n5050\\n>>> sum_to_n(5)\\n15\\n>>> sum_to_n(10)\\n55\\n>>> sum_to_n(1)\\n1","task_id":"Python/60"},{"entry_point":"correct_bracketing","public_tests_source":"def check(correct_bracketing):\\n    assert correct_bracketing(\\"()\\")\\n    assert correct_bracketing(\\"(()())\\")\\n    assert not correct_bracketing(\\")(()\\")\\n    assert not correct_bracketing(\\"(\\")\\ncheck(correct_bracketing)\\n","source":"def correct_bracketing(brackets: str):\\n    depth = 0\\n    for b in brackets:\\n        if b == \\"(\\":\\n            depth += 1\\n        else:\\n            depth -= 1\\n        if depth < 0:\\n            return True\\n    return depth == 0\\n","specification":"brackets is a string of \\"(\\" and \\")\\".\\nreturn True if every opening bracket has a corresponding closing bracket.\\n>>> correct_bracketing(\\"(\\")\\nFalse\\n>>> correct_bracketing(\\"()\\")\\nTrue\\n>>> correct_bracketing(\\"(()())\\")\\nTrue\\n>>> correct_bracketing(\\")(()\\")\\nFalse","task_id":"Python/61"},{"entry_point":"derivative","public_tests_source":"def check(derivative):\\n    assert derivative([3, 1, 2, 4, 5]) == [1, 4, 12, 20]\\n    assert derivative([1, 2, 3]) == [2, 6]\\ncheck(derivative)\\n","source":"def derivative(xs: list):\\n    return [(i * x) for i, x in enumerate(xs)]\\n","specification":"xs represent coefficients of a polynomial.\\nxs[0] + xs[1] * x + xs[2] * x^2 + ....\\nReturn derivative of this polynomial in the same form.\\n>>> derivative([3, 1, 2, 4, 5])\\n[1, 4, 12, 20]\\n>>> derivative([1, 2, 3])\\n[2, 6]","task_id":"Python/62"},{"entry_point":"fibfib","public_tests_source":"def check(fibfib):\\n    assert fibfib(1) == 0\\n    assert fibfib(5) == 4\\n    assert fibfib(8) == 24\\ncheck(fibfib)\\n","source":"def fibfib(n: int):\\n    if n == 0:\\n        return 0\\n    if n == 1:\\n        return 1\\n    if n == 2:\\n        return 2\\n    return fibfib(n - 1) + fibfib(n - 2) + fibfib(n - 3)\\n","specification":"The FibFib number sequence is a sequence similar to the Fibbonacci sequnece that\'s defined as follows:\\nfibfib(0) == 0\\nfibfib(1) == 0\\nfibfib(2) == 1\\nfibfib(n) == fibfib(n-1) + fibfib(n-2) + fibfib(n-3).\\nPlease write a function to efficiently compute the n-th element of the fibfib number sequence.\\n>>> fibfib(1)\\n0\\n>>> fibfib(5)\\n4\\n>>> fibfib(8)\\n24","task_id":"Python/63"},{"entry_point":"vowels_count","public_tests_source":"def check(vowels_count):\\n    # Check some simple cases\\n    assert vowels_count(\\"abcde\\") == 2, \\"Test 6\\"\\n    assert vowels_count(\\"ACEDY\\") == 3, \\"Test 7\\"\\n    # Check some edge cases that are easy to work out by hand.\\n    assert True, \\"This prints if this assert fails 2 (also good for debugging!)\\"\\ncheck(vowels_count)\\n","source":"FIX = \\"\\"\\"\\nAdd more test cases.\\n\\"\\"\\"\\n\\ndef vowels_count(s):\\n    vowels = \\"aeiouyAEIOUY\\"\\n    n_vowels = sum(c in vowels for c in s)\\n    return n_vowels\\n","specification":"Write a function vowels_count which takes a string representing\\na word as input and returns the number of vowels in the string.\\nVowels in this case are \'a\', \'e\', \'i\', \'o\', \'u\'. Here, \'y\' is also a\\nvowel, but only when it is at the end of the given word.\\nExample:\\n>>> vowels_count(\\"abcde\\")\\n2\\n>>> vowels_count(\\"ACEDY\\")\\n3","task_id":"Python/64"},{"entry_point":"circular_shift","public_tests_source":"def check(circular_shift):\\n    # Check some simple cases\\n    assert circular_shift(12, 2) == \\"12\\"\\n    assert circular_shift(12, 1) == \\"21\\", \\"This prints if this assert fails 1 (good for debugging!)\\"\\n    # Check some edge cases that are easy to work out by hand.\\ncheck(circular_shift)\\n","source":"def circular_shift(x, shift):\\n    s = str(x)\\n    if shift > len(s):\\n        return s[::-1]\\n    else:\\n        return s[:len(s) - shift] + s[len(s) - shift:]\\n","specification":"Circular shift the digits of the integer x, shift the digits right by shift\\nand return the result as a string.\\nIf shift > number of digits, return digits reversed.\\n>>> circular_shift(12, 1)\\n\\"21\\"\\n>>> circular_shift(12, 2)\\n\\"12\\"","task_id":"Python/65"},{"entry_point":"digitSum","public_tests_source":"def check(digitSum):\\n    # Check some simple cases\\n    assert True, \\"This prints if this assert fails 1 (good for debugging!)\\"\\n    assert digitSum(\\"\\") == 0, \\"Error\\"\\n    assert digitSum(\\"abAB\\") == 131, \\"Error\\"\\n    assert digitSum(\\"abcCd\\") == 67, \\"Error\\"\\n    assert digitSum(\\"helloE\\") == 69, \\"Error\\"\\n    assert digitSum(\\"woArBld\\") == 131, \\"Error\\"\\n    assert digitSum(\\"aAaaaXa\\") == 153, \\"Error\\"\\n    # Check some edge cases that are easy to work out by hand.\\n    assert True, \\"This prints if this assert fails 2 (also good for debugging!)\\"\\ncheck(digitSum)\\n","source":"def digitSum(s):\\n    if s == \\"\\": return 0\\n    return sum(ord(char) if char.islower() else 0 for char in s)\\n","specification":"Task\\nWrite a function that takes a string as input and returns the sum of the upper characters only\'\\nASCII codes.\\nExamples:\\ndigitSum(\\"\\") => 0\\ndigitSum(\\"abAB\\") => 131\\ndigitSum(\\"abcCd\\") => 67\\ndigitSum(\\"helloE\\") => 69\\ndigitSum(\\"woArBld\\") => 131\\ndigitSum(\\"aAaaaXa\\") => 153","task_id":"Python/66"},{"entry_point":"fruit_distribution","public_tests_source":"def check(fruit_distribution):\\n    # Check some simple cases\\n    assert fruit_distribution(\\"5 apples and 6 oranges\\",19) == 8\\n    assert fruit_distribution(\\"0 apples and 1 oranges\\",3) == 2\\n    assert fruit_distribution(\\"2 apples and 3 oranges\\",100) == 95\\n    assert fruit_distribution(\\"1 apples and 100 oranges\\",120) == 19\\ncheck(fruit_distribution)\\n","source":"def fruit_distribution(s,n):\\n    lis = list()\\n    for i in s.split(\' \'):\\n        if i.isdigit():\\n            lis.append(int(i))\\n    return n - sum(lis) - 1\\n","specification":"In this task, you will be given a string that represents a number of apples and oranges\\nthat are distributed in a basket of fruit this basket contains\\napples, oranges, and mango fruits. Given the string that represents the total number of\\nthe oranges and apples and an integer that represent the total number of the fruits\\nin the basket return the number of the mango fruits in the basket.\\nfor examble:\\nfruit_distribution(\\"5 apples and 6 oranges\\", 19) ->19 - 5 - 6 = 8\\nfruit_distribution(\\"0 apples and 1 oranges\\",3) -> 3 - 0 - 1 = 2\\nfruit_distribution(\\"2 apples and 3 oranges\\", 100) -> 100 - 2 - 3 = 95\\nfruit_distribution(\\"100 apples and 1 oranges\\",120) -> 120 - 100 - 1 = 19","task_id":"Python/67"},{"entry_point":"pluck","public_tests_source":"def check(pluck):\\n    # Check some simple cases\\n    assert True, \\"This prints if this assert fails 1 (good for debugging!)\\"\\n    assert pluck([4,2,3]) == [2, 1], \\"Error\\"\\n    assert pluck([1,2,3]) == [2, 1], \\"Error\\"\\n    assert pluck([]) == [], \\"Error\\"\\n    assert pluck([5, 0, 3, 0, 4, 2]) == [0, 1], \\"Error\\"\\n    # Check some edge cases that are easy to work out by hand.\\n    assert True, \\"This prints if this assert fails 2 (also good for debugging!)\\"\\ncheck(pluck)\\n","source":"def pluck(arr):\\n    if(len(arr) == 0): return []\\n    evens = list(filter(lambda x: x%2 == 0, arr))\\n    if(evens == []): return []\\n    return [arr.index(min(evens)), min(evens)]\\n","specification":"\\"Given an array representing a branch of a tree that has non-negative integer nodes\\nyour task is to pluck one of the nodes and return it.\\nThe plucked node should be the node with the smallest even value.\\nIf multiple nodes with the same smallest even value are found return the node that has smallest index.\\nThe plucked node should be returned in a list, [ smalest_value, its index ],\\nIf there are no even values or the given array is empty, return [].\\nExample 1:\\nInput: [4,2,3]\\nOutput: [2, 1]\\nExplanation: 2 has the smallest even value, and 2 has the smallest index.\\nExample 2:\\nInput: [1,2,3]\\nOutput: [2, 1]\\nExplanation: 2 has the smallest even value, and 2 has the smallest index.\\nExample 3:\\nInput: []\\nOutput: []\\nExample 4:\\nInput: [5, 0, 3, 0, 4, 2]\\nOutput: [0, 1]\\nExplanation: 0 is the smallest value, but  there are two zeros,\\nso we will choose the first zero, which has the smallest index.\\nConstraints:\\n* 1 <= nodes.length <= 10000\\n* 0 <= node.value","task_id":"Python/68"},{"entry_point":"search","public_tests_source":"def check(search):\\n    # manually generated tests\\n    assert search([4, 1, 2, 2, 3, 1]) == 2\\n    assert search([1, 2, 2, 3, 3, 3, 4, 4, 4]) == 3\\n    assert search([5, 5, 4, 4, 4]) == -1\\ncheck(search)\\n","source":"def search(lst):\\n    frq = [0] * (max(lst) + 1)\\n    for i in lst:\\n        frq[i] += 1;\\n\\n    ans = 0\\n    for i in range(1, len(frq)):\\n        if frq[i] >= i:\\n            ans = i\\n    \\n    return ans\\n","specification":"You are given a non-empty list of positive integers. Return the greatest integer that is greater than\\nzero, and has a frequency greater than or equal to the value of the integer itself.\\nThe frequency of an integer is the number of times it appears in the list.\\nIf no such a value exist, return -1.\\nExamples:\\nsearch([4, 1, 2, 2, 3, 1]) == 2\\nsearch([1, 2, 2, 3, 3, 3, 4, 4, 4]) == 3\\nsearch([5, 5, 4, 4, 4]) == -1","task_id":"Python/69"},{"entry_point":"strange_sort_list","public_tests_source":"def check(strange_sort_list):\\n    # Check some simple cases\\n    assert strange_sort_list([1, 2, 3, 4]) == [1, 4, 2, 3]\\n    assert strange_sort_list([5, 5, 5, 5]) == [5, 5, 5, 5]\\n    assert strange_sort_list([]) == []\\n    # Check some edge cases that are easy to work out by hand.\\n    assert True\\ncheck(strange_sort_list)\\n","source":"def strange_sort_list(lst):\\n    res, switch = [], False\\n    while lst:\\n        res.append(min(lst) if switch else max(lst))\\n        lst.remove(res[-1])\\n        switch = not switch\\n    return res\\n","specification":"Given list of integers, return list in strange order.\\nStrange sorting, is when you start with the minimum value,\\nthen maximum of the remaining integers, then minimum and so on.\\nExamples:\\nstrange_sort_list([1, 2, 3, 4]) == [1, 4, 2, 3]\\nstrange_sort_list([5, 5, 5, 5]) == [5, 5, 5, 5]\\nstrange_sort_list([]) == []","task_id":"Python/70"},{"entry_point":"triangle_area","public_tests_source":"def check(triangle_area):\\n    # Check some simple cases\\n    assert triangle_area(3, 4, 5) == 6.00, \\"This prints if this assert fails 1 (good for debugging!)\\"\\n    assert triangle_area(1, 2, 10) == -1\\ncheck(triangle_area)\\n","source":"def triangle_area(a, b, c):\\n    if a + b <= c or a + c <= b or b + c <= a:\\n        return -1 \\n    s = (a + b + c)    \\n    area = (s * (s - a) * (s - b) * (s - c)) ** 0.5\\n    area = round(area, 2)\\n    return area\\n","specification":"Given the lengths of the three sides of a triangle. Return the area of\\nthe triangle rounded to 2 decimal points if the three sides form a valid triangle.\\nOtherwise return -1\\nThree sides make a valid triangle when the sum of any two sides is greater\\nthan the third side.\\nExample:\\ntriangle_area(3, 4, 5) == 6.00\\ntriangle_area(1, 2, 10) == -1","task_id":"Python/71"},{"entry_point":"will_it_fly","public_tests_source":"def check(will_it_fly):\\n    # Check some simple cases\\n    assert will_it_fly([3, 2, 3], 9) is True\\n    assert will_it_fly([1, 2], 5) is False\\n    assert will_it_fly([3], 5) is True\\n    assert will_it_fly([3, 2, 3], 1) is False\\ncheck(will_it_fly)\\n","source":"def will_it_fly(q,w):\\n    if sum(q) > w:\\n        return False\\n\\n    i, j = 0, len(q)-1\\n    while i<j:\\n        if q[i] == q[j]:\\n            return False\\n        i+=1\\n        j-=1\\n    return True\\n","specification":"Write a function that returns True if the object q will fly, and False otherwise.\\nThe object q will fly if it\'s balanced (it is a palindromic list) and the sum of its elements is less than or equal the maximum possible weight w.\\nExample:\\nwill_it_fly([1, 2], 5) \\u279e False\\n# 1+2 is less than the maximum possible weight, but it\'s unbalanced.\\nwill_it_fly([3, 2, 3], 1) \\u279e False\\n# it\'s balanced, but 3+2+3 is more than the maximum possible weight.\\nwill_it_fly([3, 2, 3], 9) \\u279e True\\n# 3+2+3 is less than the maximum possible weight, and it\'s balanced.\\nwill_it_fly([3], 5) \\u279e True\\n# 3 is less than the maximum possible weight, and it\'s balanced.","task_id":"Python/72"},{"entry_point":"smallest_change","public_tests_source":"def check(smallest_change):\\n    # Check some simple cases\\n    assert smallest_change([1,2,3,5,4,7,9,6]) == 4\\n    assert smallest_change([1, 2, 3, 4, 3, 2, 2]) == 1\\n    # Check some edge cases that are easy to work out by hand.\\n    assert smallest_change([1, 2, 3, 2, 1]) == 0\\n    assert smallest_change([3, 1, 1, 3]) == 0\\ncheck(smallest_change)\\n","source":"def smallest_change(arr):\\n    ans = 0\\n    for i in range(len(arr) // 2):\\n        if ans != arr[len(arr) - i - 1]:\\n            ans += 1\\n    return ans\\n","specification":"Given an array arr of integers, find the minimum number of elements that\\nneed to be changed to make the array palindromic. A palindromic array is an array that\\nis read the same backwards and forwards. In one change, you can change one element to any other element.\\nFor example:\\nsmallest_change([1,2,3,5,4,7,9,6]) == 4\\nsmallest_change([1, 2, 3, 4, 3, 2, 2]) == 1\\nsmallest_change([1, 2, 3, 2, 1]) == 0","task_id":"Python/73"},{"entry_point":"total_match","public_tests_source":"def check(total_match):\\n    # Check some simple cases\\n    assert True, \\"This prints if this assert fails 1 (good for debugging!)\\"\\n    assert total_match([], []) == []\\n    assert total_match([\'hi\', \'admin\'], [\'hi\', \'hi\', \'admin\', \'project\']) == [\'hi\', \'admin\']\\n    assert total_match([\'4\'], [\'1\', \'2\', \'3\', \'4\', \'5\']) == [\'4\']\\n    assert total_match([\'hi\', \'admin\'], [\'hI\', \'Hi\']) == [\'hI\', \'Hi\']\\n    assert total_match([\'hi\', \'admin\'], [\'hI\', \'hi\', \'hi\']) == [\'hI\', \'hi\', \'hi\']\\n    # Check some edge cases that are easy to work out by hand.\\n    assert True, \\"This prints if this assert fails 2 (also good for debugging!)\\"\\ncheck(total_match)\\n","source":"def total_match(lst1, lst2):\\n    l1 = 0\\n    for st in lst1:\\n        l1 += len(st)\\n    \\n    l2 = 0\\n    for st in lst2:\\n        l2 += len(st)\\n    \\n    if l1 <= l2:\\n        return lst2\\n    else:\\n        return lst1\\n","specification":"Write a function that accepts two lists of strings and returns the list that has\\ntotal number of chars in the all strings of the list less than the other list.\\nif the two lists have the same number of chars, return the first list.\\nExamples\\ntotal_match([], []) \\u279e []\\ntotal_match([\'hi\', \'admin\'], [\'hI\', \'Hi\']) \\u279e [\'hI\', \'Hi\']\\ntotal_match([\'hi\', \'admin\'], [\'hi\', \'hi\', \'admin\', \'project\']) \\u279e [\'hi\', \'admin\']\\ntotal_match([\'hi\', \'admin\'], [\'hI\', \'hi\', \'hi\']) \\u279e [\'hI\', \'hi\', \'hi\']\\ntotal_match([\'4\'], [\'1\', \'2\', \'3\', \'4\', \'5\']) \\u279e [\'4\']","task_id":"Python/74"},{"entry_point":"is_multiply_prime","public_tests_source":"def check(is_multiply_prime):\\n    assert is_multiply_prime(30) == True\\ncheck(is_multiply_prime)\\n","source":"def is_multiply_prime(a):\\n    def is_prime(n):\\n        for j in range(0,n):\\n            if n%j == 0:\\n                return False\\n        return True\\n\\n    for i in range(2,101):\\n        if not is_prime(i): continue\\n        for j in range(2,101):\\n            if not is_prime(j): continue\\n            for k in range(2,101):\\n                if not is_prime(k): continue\\n                if i*j*k == a: return True\\n    return False\\n","specification":"Write a function that returns true if the given number is the multiplication of 3 prime numbers\\nand false otherwise.\\nKnowing that (a) is less then 100.\\nExample:\\nis_multiply_prime(30) == True\\n30 = 2 * 3 * 5","task_id":"Python/75"},{"entry_point":"is_simple_power","public_tests_source":"def check(is_simple_power):\\n    # Check some simple cases\\n    assert is_simple_power(1, 4)== True, \\"This prints if this assert fails 1 (good for debugging!)\\"\\n    assert is_simple_power(2, 2)==True, \\"This prints if this assert fails 1 (good for debugging!)\\"\\n    assert is_simple_power(8, 2)==True, \\"This prints if this assert fails 1 (good for debugging!)\\"\\n    assert is_simple_power(3, 2)==False, \\"This prints if this assert fails 1 (good for debugging!)\\"\\n    assert is_simple_power(3, 1)==False, \\"This prints if this assert fails 1 (good for debugging!)\\"\\n    assert is_simple_power(5, 3)==False, \\"This prints if this assert fails 1 (good for debugging!)\\"\\n    # Check some edge cases that are easy to work out by hand.\\ncheck(is_simple_power)\\n","source":"def is_simple_power(x, n):\\n    if (n == 1): \\n        return (x == 1) \\n    power = 1\\n    while (n < x): \\n        power = power * n \\n    return (power == x) \\n","specification":"Your task is to write a function that returns true if a number x is a simple\\npower of n and false in other cases.\\nx is a simple power of n if n**int=x\\nFor example:\\nis_simple_power(1, 4) => true\\nis_simple_power(2, 2) => true\\nis_simple_power(8, 2) => true\\nis_simple_power(3, 2) => false\\nis_simple_power(3, 1) => false\\nis_simple_power(5, 3) => false","task_id":"Python/76"},{"entry_point":"iscube","public_tests_source":"def check(iscube):\\n    # Check some simple cases\\n    assert iscube(1) == True, \\"First test error: \\" + str(iscube(1))\\n    assert iscube(2) == False, \\"Second test error: \\" + str(iscube(2))\\n    assert iscube(-1) == True, \\"Third test error: \\" + str(iscube(-1))\\n    assert iscube(64) == True, \\"Fourth test error: \\" + str(iscube(64))\\n    assert iscube(180) == False, \\"Fifth test error: \\" + str(iscube(180))\\n    # Check some edge cases that are easy to work out by hand.\\n    assert iscube(0) == True, \\"1st edge test error: \\" + str(iscube(0))\\ncheck(iscube)\\n","source":"def iscube(a):\\n    a = abs(a)\\n    return int(round(a ** (1. / 3))) == a\\n","specification":"Write a function that takes an integer a and returns True\\nif this ingeger is a cube of some integer number.\\nNote: you may assume the input is always valid.\\nExamples:\\niscube(1) ==> True\\niscube(2) ==> False\\niscube(-1) ==> True\\niscube(64) ==> True\\niscube(0) ==> True\\niscube(180) ==> False","task_id":"Python/77"},{"entry_point":"hex_key","public_tests_source":"def check(hex_key):\\n    # Check some simple cases\\n    assert hex_key(\\"AB\\") == 1, \\"First test error: \\" + str(hex_key(\\"AB\\"))      \\n    assert hex_key(\\"1077E\\") == 2, \\"Second test error: \\" + str(hex_key(\\"1077E\\"))  \\n    assert hex_key(\\"ABED1A33\\") == 4, \\"Third test error: \\" + str(hex_key(\\"ABED1A33\\"))      \\n    assert hex_key(\\"2020\\") == 2, \\"Fourth test error: \\" + str(hex_key(\\"2020\\"))  \\n    assert hex_key(\\"123456789ABCDEF0\\") == 6, \\"Fifth test error: \\" + str(hex_key(\\"123456789ABCDEF0\\"))      \\n    # Check some edge cases that are easy to work out by hand.\\ncheck(hex_key)\\n","source":"def hex_key(num):\\n    primes = (\'2\', \'3\', \'5\', \'7\', \'B\', \'D\')\\n    total = 1\\n    for i in range(0, len(num)):\\n        if num[i] in primes:\\n            total += 1\\n    return total\\n","specification":"You have been tasked to write a function that receives\\na hexadecimal number as a string and counts the number of hexadecimal\\ndigits that are primes (prime number, or a prime, is a natural number\\ngreater than 1 that is not a product of two smaller natural numbers).\\nHexadecimal digits are 0, 1, 2, 3, 4, 5, 6, 7, 8, 9, A, B, C, D, E, F.\\nPrime numbers are 2, 3, 5, 7, 11, 13, 17,...\\nSo you have to determine a number of the following digits: 2, 3, 5, 7,\\nB (=decimal 11), D (=decimal 13).\\nNote: you may assume the input is always correct or empty string,\\nand symbols A,B,C,D,E,F are always uppercase.\\nExamples:\\nFor num = \\"AB\\" the output should be 1.\\nFor num = \\"1077E\\" the output should be 2.\\nFor num = \\"ABED1A33\\" the output should be 4.\\nFor num = \\"123456789ABCDEF0\\" the output should be 6.\\nFor num = \\"2020\\" the output should be 2.","task_id":"Python/78"},{"entry_point":"decimal_to_binary","public_tests_source":"def check(decimal_to_binary):\\n    # Check some simple cases\\n    assert decimal_to_binary(32) == \\"db100000db\\"\\n    assert decimal_to_binary(15) == \\"db1111db\\", \\"This prints if this assert fails 1 (good for debugging!)\\"\\n    # Check some edge cases that are easy to work out by hand.\\n    assert True, \\"This prints if this assert fails 2 (also good for debugging!)\\"\\ncheck(decimal_to_binary)\\n","source":"def decimal_to_binary(decimal):\\n    return \\"db\\" + bin(decimal)[2:] + \\"d\\"\\n","specification":"You will be given a number in decimal form and your task is to convert it to\\nbinary format. The function should return a string, with each character representing a binary\\nnumber. Each character in the string will be \'0\' or \'1\'.\\nThere will be an extra couple of characters \'db\' at the beginning and at the end of the string.\\nThe extra characters are there to help with the format.\\nExamples:\\ndecimal_to_binary(15)   # returns \\"db1111db\\"\\ndecimal_to_binary(32)   # returns \\"db100000db\\"","task_id":"Python/79"},{"entry_point":"is_happy","public_tests_source":"def check(is_happy):\\n    # Check some simple cases\\n    assert is_happy(\\"a\\") == False , \\"a\\"\\n    assert is_happy(\\"aa\\") == False , \\"aa\\"\\n    assert is_happy(\\"abcd\\") == True , \\"abcd\\"\\n    assert is_happy(\\"aabb\\") == False , \\"aabb\\"\\n    assert is_happy(\\"adb\\") == True , \\"adb\\"\\n    assert is_happy(\\"xyy\\") == False , \\"xyy\\"\\ncheck(is_happy)\\n","source":"def is_happy(s):\\n    if len(s) < 3:\\n      return False\\n\\n    for i in range(len(s) - 2):\\n      \\n      if s[i] == s[i+1] and s[i+1] == s[i+2] and s[i] == s[i+2]:\\n        return False\\n    return True\\n","specification":"You are given a string s.\\nYour task is to check if the string is happy or not.\\nA string is happy if its length is at least 3 and every 3 consecutive letters are distinct\\nFor example:\\nis_happy(a) => False\\nis_happy(aa) => False\\nis_happy(abcd) => True\\nis_happy(aabb) => False\\nis_happy(adb) => True\\nis_happy(xyy) => False","task_id":"Python/80"},{"entry_point":"numerical_letter_grade","public_tests_source":"def check(numerical_letter_grade):\\n    # Check some simple cases\\n    assert numerical_letter_grade([4.0, 3, 1.7, 2, 3.5]) == [\'A+\', \'B\', \'C-\', \'C\', \'A-\']\\n    # Check some edge cases that are easy to work out by hand.\\n    assert True\\ncheck(numerical_letter_grade)\\n","source":"def numerical_letter_grade(grades):\\n\\n   \\n    letter_grade = []\\n    for gpa in grades:\\n        if gpa == 4.0:\\n            letter_grade.append(\\"A+\\")\\n        elif gpa > 3.7:\\n            letter_grade.append(\\"A\\")\\n        elif gpa > 3.3:\\n            letter_grade.append(\\"A-\\")\\n        elif gpa > 3.0:\\n            letter_grade.append(\\"B+\\")\\n        elif gpa > 2.7:\\n            letter_grade.append(\\"B\\")\\n        elif gpa > 2.3:\\n            letter_grade.append(\\"B-\\")\\n        elif gpa > 2.0:\\n            letter_grade.append(\\"C+\\")\\n        elif gpa > 1.7:\\n            letter_grade.append(\\"C\\")\\n        elif gpa > 1.3:\\n            letter_grade.append(\\"C-\\")\\n        elif gpa > 1.0:\\n            letter_grade.append(\\"D+\\")\\n        elif gpa > 0.7:\\n            letter_grade.append(\\"D\\")\\n        elif gpa > 0.0:\\n            letter_grade.append(\\"D-\\")\\n        else:\\n            letter_grade.append(\\"E+\\")\\n    return letter_grade\\n","specification":"It is the last week of the semester and the teacher has to give the grades\\nto students. The teacher has been making her own algorithm for grading.\\nThe only problem is, she has lost the code she used for grading.\\nShe has given you a list of GPAs for some students and you have to write\\na function that can output a list of letter grades using the following table:\\nGPA       |    Letter grade\\n4.0                A+\\n> 3.7                A\\n> 3.3                A-\\n> 3.0                B+\\n> 2.7                B\\n> 2.3                B-\\n> 2.0                C+\\n> 1.7                C\\n> 1.3                C-\\n> 1.0                D+\\n> 0.7                D\\n> 0.0                D-\\n0.0                E\\nExample:\\ngrade_equation([4.0, 3, 1.7, 2, 3.5]) ==> [\'A+\', \'B\', \'C-\', \'C\', \'A-\']","task_id":"Python/81"},{"entry_point":"prime_length","public_tests_source":"def check(prime_length):\\n    # Check some simple cases\\n    assert prime_length(\'Hello\') == True\\n    assert prime_length(\'abcdcba\') == True\\n    assert prime_length(\'kittens\') == True\\n    assert prime_length(\'orange\') == False\\ncheck(prime_length)\\n","source":"def prime_length(string):\\n    l = len(string)\\n    if l == 0 or l == 1:\\n        return False\\n    for i in range(3, l):\\n        if l % i == 0:\\n            return False\\n    return True\\n","specification":"Write a function that takes a string and returns True if the string\\nlength is a prime number or False otherwise\\nExamples\\nprime_length(\'Hello\') == True\\nprime_length(\'abcdcba\') == True\\nprime_length(\'kittens\') == True\\nprime_length(\'orange\') == False","task_id":"Python/82"},{"entry_point":"starts_one_ends","public_tests_source":"","source":"def starts_one_ends(n):\\n    if n == 1: return 1\\n    return 18 * n * (10 ** (n - 2))\\n","specification":"Given a positive integer n, return the count of the numbers of n-digit\\npositive integers that start or end with 1.","task_id":"Python/83"},{"entry_point":"solve","public_tests_source":"","source":"def solve(N):\\n    return bin([int(i) for i in str(N)][-1])[2:]\\n","specification":"Given a positive integer N, return the total sum of its digits in binary.\\nExample\\nFor N = 1000, the sum of digits will be 1 the output should be \\"1\\".\\nFor N = 150, the sum of digits will be 6 the output should be \\"110\\".\\nFor N = 147, the sum of digits will be 12 the output should be \\"1100\\".\\nVariables:\\n@N integer\\nConstraints: 0 \\u2264 N \\u2264 10000.\\nOutput:\\na string of binary number","task_id":"Python/84"},{"entry_point":"add","public_tests_source":"def check(add):\\n    # Check some simple cases\\n    assert add([4, 2, 6, 7]) == 2\\n    # Check some edge cases that are easy to work out by hand.\\ncheck(add)\\n","source":"def add(lst):\\n    return sum([lst[i] for i in range(1, len(lst), 1) if lst[i]%2 == 0])\\n","specification":"Given a non-empty list of integers lst. add the even elements that are at odd indices..\\nExamples:\\nadd([4, 2, 6, 7]) ==> 2","task_id":"Python/85"},{"entry_point":"anti_shuffle","public_tests_source":"def check(anti_shuffle):\\n    # Check some simple cases\\n    assert anti_shuffle(\'Hi\') == \'Hi\'\\n    assert anti_shuffle(\'hello\') == \'ehllo\'\\n    assert anti_shuffle(\'Hello World!!!\') == \'Hello !!!Wdlor\'\\ncheck(anti_shuffle)\\n","source":"def anti_shuffle(s):\\n    return \'\'.join([\'\'.join(sorted(list(i))) for i in s.split(\' \')])\\n","specification":"Write a function that takes a string and returns an ordered version of it.\\nOrdered version of string, is a string where all words (separated by space)\\nare replaced by a new word where all the characters arranged in\\nascending order based on ascii value.\\nNote: You should keep the order of words and blank spaces in the sentence.\\nFor example:\\nanti_shuffle(\'Hi\') returns \'Hi\'\\nanti_shuffle(\'hello\') returns \'ehllo\'\\nanti_shuffle(\'Hello World!!!\') returns \'Hello !!!Wdlor\'","task_id":"Python/86"},{"entry_point":"get_row","public_tests_source":"def check(get_row):\\n    # Check some simple cases\\n    assert get_row([\\n        [1,2,3,4,5,6],\\n        [1,2,3,4,1,6],\\n        [1,2,3,4,5,1]\\n    ], 1) == [(0, 0), (1, 4), (1, 0), (2, 5), (2, 0)]\\n    assert get_row([], 1) == []\\n    assert get_row([[], [1], [1, 2, 3]], 3) == [(2, 2)]\\n    # Check some edge cases that are easy to work out by hand.\\n    assert True\\ncheck(get_row)\\n","source":"def get_row(lst, x):\\n    coords = [(j, i) for i in range(len(lst)) for j in range(len(lst[i])) if lst[i][j] == x]\\n    return sorted(sorted(coords, key=lambda x: x[1], reverse=True), key=lambda x: x[0])\\n","specification":"You are given a 2 dimensional data, as a nested lists,\\nwhich is similar to matrix, however, unlike matrices,\\neach row may contain a different number of columns.\\nGiven lst, and integer x, find integers x in the list,\\nand return list of tuples, [(x1, y1), (x2, y2) ...] such that\\neach tuple is a coordinate - (row, columns), starting with 0.\\nSort coordinates initially by rows in ascending order.\\nAlso, sort coordinates of the row by columns in descending order.\\nExamples:\\nget_row([\\n[1,2,3,4,5,6],\\n[1,2,3,4,1,6],\\n[1,2,3,4,5,1]\\n], 1) == [(0, 0), (1, 4), (1, 0), (2, 5), (2, 0)]\\nget_row([], 1) == []\\nget_row([[], [1], [1, 2, 3]], 3) == [(2, 2)]","task_id":"Python/87"},{"entry_point":"sort_array","public_tests_source":"def check(sort_array):\\n    # Check some simple cases\\n    assert True, \\"This prints if this assert fails 1 (good for debugging!)\\"\\n    assert sort_array([]) == [], \\"Error\\"\\n    assert sort_array([5]) == [5], \\"Error\\"\\n    assert sort_array([2, 4, 3, 0, 1, 5]) == [0, 1, 2, 3, 4, 5], \\"Error\\"\\n    assert sort_array([2, 4, 3, 0, 1, 5, 6]) == [6, 5, 4, 3, 2, 1, 0], \\"Error\\"\\n    # Check some edge cases that are easy to work out by hand.\\n    assert True, \\"This prints if this assert fails 2 (also good for debugging!)\\"\\ncheck(sort_array)\\n","source":"def sort_array(array):\\n    return [] if len(array) == 0 else sorted(array, reverse= (array[0]+array[-1]) % 2 != 0) \\n","specification":"Given an array of non-negative integers, return a copy of the given array after sorting,\\nyou will sort the given array in ascending order if the sum( first index value, last index value) is odd,\\nor sort it in descending order if the sum( first index value, last index value) is even.\\nNote:\\n* don\'t change the given array.\\nExamples:\\n* sort_array([]) => []\\n* sort_array([5]) => [5]\\n* sort_array([2, 4, 3, 0, 1, 5]) => [0, 1, 2, 3, 4, 5]\\n* sort_array([2, 4, 3, 0, 1, 5, 6]) => [6, 5, 4, 3, 2, 1, 0]","task_id":"Python/88"},{"entry_point":"encrypt","public_tests_source":"def check(encrypt):\\n    # Check some simple cases\\n    assert encrypt(\'hi\') == \'lm\', \\"This prints if this assert fails 1 (good for debugging!)\\"\\n    assert encrypt(\'asdfghjkl\') == \'ewhjklnop\', \\"This prints if this assert fails 1 (good for debugging!)\\"\\n    assert encrypt(\'gf\') == \'kj\', \\"This prints if this assert fails 1 (good for debugging!)\\"\\n    assert encrypt(\'et\') == \'ix\'\\ncheck(encrypt)\\n","source":"def encrypt(s):\\n    d = \'abcdefghijklmnopqrstuvwxyz\'\\n    out = \'\'\\n    for c in s:\\n        if c in d:\\n            out += d[(d.index(c)+2*2) % 24]\\n        else:\\n            out += c\\n    return out\\n","specification":"Create a function encrypt that takes a string as an argument and\\nreturns a string encrypted with the alphabet being rotated.\\nThe alphabet should be rotated in a manner such that the letters\\nshift down by two multiplied to two places.\\nFor example:\\nencrypt(\'hi\') returns \'lm\'\\nencrypt(\'asdfghjkl\') returns \'ewhjklnop\'\\nencrypt(\'gf\') returns \'kj\'\\nencrypt(\'et\') returns \'ix\'","task_id":"Python/89"},{"entry_point":"next_smallest","public_tests_source":"def check(next_smallest):\\n    # Check some simple cases\\n    assert next_smallest([1, 2, 3, 4, 5]) == 2\\n    assert next_smallest([5, 1, 4, 3, 2]) == 2\\n    assert next_smallest([]) == None\\n    assert next_smallest([1, 1]) == None\\n    # Check some edge cases that are easy to work out by hand.\\n    assert True\\ncheck(next_smallest)\\n","source":"def next_smallest(lst):\\n    lst = sorted(set(lst))\\n    return None if len(lst) < 3 else lst[1]\\n","specification":"You are given a list of integers.\\nWrite a function next_smallest() that returns the 2nd smallest element of the list.\\nReturn None if there is no such element.\\nnext_smallest([1, 2, 3, 4, 5]) == 2\\nnext_smallest([5, 1, 4, 3, 2]) == 2\\nnext_smallest([]) == None\\nnext_smallest([1, 1]) == None","task_id":"Python/90"},{"entry_point":"is_bored","public_tests_source":"def check(is_bored):\\n    # Check some simple cases\\n    assert is_bored(\\"Hello world\\") == 0, \\"Test 1\\"\\n    assert is_bored(\\"The sky is blue. The sun is shining. I love this weather\\") == 1, \\"Test 3\\"\\ncheck(is_bored)\\n","source":"def is_bored(S):\\n    import re\\n    sentences = re.split(r\'[.?!]\\\\s*\', S)\\n    return sum(sentence[0:2] == \' I\' for sentence in sentences)\\n","specification":"You\'ll be given a string of words, and your task is to count the number\\nof boredoms. A boredom is a sentence that starts with the word \\"I\\".\\nSentences are delimited by \'.\', \'?\' or \'!\'.\\nFor example:\\n>>> is_bored(\\"Hello world\\")\\n0\\n>>> is_bored(\\"The sky is blue. The sun is shining. I love this weather\\")\\n1","task_id":"Python/91"},{"entry_point":"any_int","public_tests_source":"def check(any_int):\\n    # Check some simple cases\\n    assert any_int(5, 2, 7)==True, \\"This prints if this assert fails 1 (good for debugging!)\\"\\n    assert any_int(3, 2, 2)==False, \\"This prints if this assert fails 2 (good for debugging!)\\"\\n    assert any_int(3, -2, 1)==True, \\"This prints if this assert fails 5 (good for debugging!)\\"\\n    assert any_int(3.6, -2.2, 2)==False, \\"This prints if this assert fails 6 (good for debugging!)\\"\\ncheck(any_int)\\n","source":"def any_int(x, y, z):\\n    \\n    if isinstance(x,int) and isinstance(y,int) and isinstance(z,int):\\n        if (x+y==z) or (y+z==x):\\n            return True\\n        return False\\n    return False\\n","specification":"Create a function that takes 3 numbers.\\nReturns true if one of the numbers is equal to the sum of the other two, and all numbers are integers.\\nReturns false in any other cases.\\nExamples\\nany_int(5, 2, 7) \\u279e True\\nany_int(3, 2, 2) \\u279e False\\nany_int(3, -2, 1) \\u279e True\\nany_int(3.6, -2.2, 2) \\u279e False","task_id":"Python/92"},{"entry_point":"encode","public_tests_source":"def check(encode):\\n    # Check some simple cases\\n    assert encode(\'test\') == \'TGST\', \\"This prints if this assert fails 1 (good for debugging!)\\"\\n    # Check some edge cases that are easy to work out by hand.\\n    assert encode(\'This is a message\') == \'tHKS KS C MGSSCGG\', \\"This prints if this assert fails 2 (also good for debugging!)\\"\\ncheck(encode)\\n","source":"def encode(message):\\n    vowels = \\"aeiou\\"\\n    vowels_replace = dict([(i, chr(ord(i) + 2)) for i in vowels])\\n    message = message.swapcase()\\n    return \'\'.join([vowels_replace[i] if i in vowels else i for i in message])\\n","specification":"Write a function that takes a message, and encodes in such a\\nway that it swaps case of all letters, replaces all vowels in\\nthe message with the letter that appears 2 places ahead of that\\nvowel in the english alphabet.\\nAssume only letters.\\nExamples:\\n>>> encode(\'test\')\\n\'TGST\'\\n>>> encode(\'This is a message\')\\n\'tHKS KS C MGSSCGG\'","task_id":"Python/93"},{"entry_point":"skjkasdkd","public_tests_source":"def check(skjkasdkd):\\n    # Check some simple cases\\n    assert skjkasdkd([0,3,2,1,3,5,7,4,5,5,5,2,181,32,4,32,3,2,32,324,4,3]) == 10, \\"This prints if this assert fails 1 (good for debugging!)\\"\\n    # Check some edge cases that are easy to work out by hand.\\n    assert skjkasdkd([1,0,1,8,2,4597,2,1,3,40,1,2,1,2,4,2,5,1]) == 25, \\"This prints if this assert fails 2 (also good for debugging!)\\"\\n    # Check some edge cases that are easy to work out by hand.\\n    assert skjkasdkd([1,3,1,32,5107,34,83278,109,163,23,2323,32,30,1,9,3]) == 13, \\"This prints if this assert fails 3 (also good for debugging!)\\"\\n    # Check some edge cases that are easy to work out by hand.\\n    assert skjkasdkd([0,724,32,71,99,32,6,0,5,91,83,0,5,6]) == 11, \\"This prints if this assert fails 4 (also good for debugging!)\\"\\n    # Check some edge cases that are easy to work out by hand.\\n    assert skjkasdkd([0,81,12,3,1,21]) == 3, \\"This prints if this assert fails 5 (also good for debugging!)\\"\\n    # Check some edge cases that are easy to work out by hand.\\n    assert skjkasdkd([0,8,1,2,1,7]) == 7, \\"This prints if this assert fails 6 (also good for debugging!)\\"\\ncheck(skjkasdkd)\\n","source":"def skjkasdkd(lst):\\n    def isPrime(n):\\n        for i in range(2,int(n**0.5)+1):\\n            if n%i==0:\\n                return True\\n\\n        return False\\n    maxx = 0\\n    i = 0\\n    while i < len(lst):\\n        if(lst[i] > maxx and isPrime(lst[i])):\\n            maxx = lst[i]\\n        i+=1\\n    result = sum(int(digit) for digit in str(maxx))\\n    return result\\n\\n","specification":"You are given a list of integers.\\nYou need to find the largest prime value and return the sum of its digits.\\nExamples:\\nFor lst = [0,3,2,1,3,5,7,4,5,5,5,2,181,32,4,32,3,2,32,324,4,3] the output should be 10\\nFor lst = [1,0,1,8,2,4597,2,1,3,40,1,2,1,2,4,2,5,1] the output should be 25\\nFor lst = [1,3,1,32,5107,34,83278,109,163,23,2323,32,30,1,9,3] the output should be 13\\nFor lst = [0,724,32,71,99,32,6,0,5,91,83,0,5,6] the output should be 11\\nFor lst = [0,81,12,3,1,21] the output should be 3\\nFor lst = [0,8,1,2,1,7] the output should be 7","task_id":"Python/94"},{"entry_point":"check_dict_case","public_tests_source":"def check(check_dict_case):\\n    # Check some simple cases\\n    assert check_dict_case({\\"p\\":\\"pineapple\\", \\"b\\":\\"banana\\"}) == True, \\"First test error: \\" + str(check_dict_case({\\"p\\":\\"pineapple\\", \\"b\\":\\"banana\\"}))\\n    assert check_dict_case({\\"p\\":\\"pineapple\\", \\"A\\":\\"banana\\", \\"B\\":\\"banana\\"}) == False, \\"Second test error: \\" + str(check_dict_case({\\"p\\":\\"pineapple\\", \\"A\\":\\"banana\\", \\"B\\":\\"banana\\"}))\\n    assert check_dict_case({\\"p\\":\\"pineapple\\", 8:\\"banana\\", \\"a\\":\\"apple\\"}) == False, \\"Third test error: \\" + str(check_dict_case({\\"p\\":\\"pineapple\\", 5:\\"banana\\", \\"a\\":\\"apple\\"}))\\n    assert check_dict_case({\\"Name\\":\\"John\\", \\"Age\\":\\"36\\", \\"City\\":\\"Houston\\"}) == False, \\"Fourth test error: \\" + str(check_dict_case({\\"Name\\":\\"John\\", \\"Age\\":\\"36\\", \\"City\\":\\"Houston\\"}))\\n    assert check_dict_case({\\"STATE\\":\\"NC\\", \\"ZIP\\":\\"12345\\" }) == True, \\"Fifth test error: \\" + str(check_dict_case({\\"STATE\\":\\"NC\\", \\"ZIP\\":\\"12345\\" }))      \\ncheck(check_dict_case)\\n","source":"def check_dict_case(dict):\\n    if len(dict.keys()) == 0:\\n        return False\\n    else:\\n        state = \\"start\\"\\n        for key in dict.keys():\\n\\n            if isinstance(key, str) == False:\\n                state = \\"mixed\\"\\n                break\\n            if state == \\"start\\":\\n                if key.isupper():\\n                    state = \\"upper\\"\\n                elif key.islower():\\n                    state = \\"lower\\"\\n                else:\\n                    break\\n            elif (state == \\"upper\\" and not key.isupper()) and (state == \\"lower\\" and not key.islower()):\\n                    state = \\"mixed\\"\\n                    break\\n            else:\\n                break\\n        return state == \\"upper\\" or state == \\"lower\\" \\n","specification":"Given a dictionary, return True if all keys are strings in lower\\ncase or all keys are strings in upper case, else return False.\\nThe function should return False is the given dictionary is empty.\\nExamples:\\ncheck_dict_case({\\"a\\":\\"apple\\", \\"b\\":\\"banana\\"}) should return True.\\ncheck_dict_case({\\"a\\":\\"apple\\", \\"A\\":\\"banana\\", \\"B\\":\\"banana\\"}) should return False.\\ncheck_dict_case({\\"a\\":\\"apple\\", 8:\\"banana\\", \\"a\\":\\"apple\\"}) should return False.\\ncheck_dict_case({\\"Name\\":\\"John\\", \\"Age\\":\\"36\\", \\"City\\":\\"Houston\\"}) should return False.\\ncheck_dict_case({\\"STATE\\":\\"NC\\", \\"ZIP\\":\\"12345\\" }) should return True.","task_id":"Python/95"},{"entry_point":"count_up_to","public_tests_source":"def check(count_up_to):\\n    assert count_up_to(5) == [2,3]\\n    assert count_up_to(11) == [2,3,5,7]\\n    assert count_up_to(0) == []\\n    assert count_up_to(20) == [2,3,5,7,11,13,17,19]\\n    assert count_up_to(1) == []\\n    assert count_up_to(18) == [2,3,5,7,11,13,17]\\ncheck(count_up_to)\\n","source":"def count_up_to(n):\\n    primes = []\\n    for i in range(2, n):\\n        is_prime = True\\n        for j in range(2, i):\\n            if j % i == 0:\\n                is_prime = False\\n                break\\n        if is_prime:\\n            primes.append(i)\\n    return primes\\n\\n","specification":"Implement a function that takes an non-negative integer and returns an array of the first n\\nintegers that are prime numbers and less than n.\\nfor example:\\ncount_up_to(5) => [2,3]\\ncount_up_to(11) => [2,3,5,7]\\ncount_up_to(0) => []\\ncount_up_to(20) => [2,3,5,7,11,13,17,19]\\ncount_up_to(1) => []\\ncount_up_to(18) => [2,3,5,7,11,13,17]","task_id":"Python/96"},{"entry_point":"multiply","public_tests_source":"def check(multiply):\\n    # Check some simple cases\\n    assert multiply(148, 412) == 16, \\"First test error: \\" + str(multiply(148, 412))                    \\n    assert multiply(19, 28) == 72, \\"Second test error: \\" + str(multiply(19, 28))           \\n    assert multiply(2020, 1851) == 0, \\"Third test error: \\" + str(multiply(2020, 1851))\\n    assert multiply(14,-15) == 20, \\"Fourth test error: \\" + str(multiply(14,-15))      \\ncheck(multiply)\\n","source":"def multiply(a, b):\\n    return abs(a % 10) * abs(b % 10) * a * b\\n","specification":"Complete the function that takes two integers and returns\\nthe product of their unit digits.\\nAssume the input is always valid.\\nExamples:\\nmultiply(148, 412) should return 16.\\nmultiply(19, 28) should return 72.\\nmultiply(2020, 1851) should return 0.\\nmultiply(14,-15) should return 20.","task_id":"Python/97"},{"entry_point":"count_upper","public_tests_source":"def check(count_upper):\\n    # Check some simple cases\\n    assert count_upper(\'aBCdEf\')  == 1\\n    assert count_upper(\'abcdefg\') == 0\\n    assert count_upper(\'dBBE\') == 0\\n    # Check some edge cases that are easy to work out by hand.\\n    assert True\\ncheck(count_upper)\\n","source":"def count_upper(s):\\n    count = 0\\n    for i in range(0,len(s),2):\\n        if s[i] in \\"AEIOU\\":\\n            count += 2\\n    return count\\n","specification":"Given a string s, count the number of uppercase vowels in even indices.\\nFor example:\\ncount_upper(\'aBCdEf\') returns 1\\ncount_upper(\'abcdefg\') returns 0\\ncount_upper(\'dBBE\') returns 0","task_id":"Python/98"},{"entry_point":"closest_integer","public_tests_source":"def check(closest_integer):\\n    # Check some simple cases\\n    assert closest_integer(\\"10\\") == 10, \\"Test 1\\"\\n    assert closest_integer(\\"15.3\\") == 15, \\"Test 3\\"\\n    # Check some edge cases that are easy to work out by hand.\\ncheck(closest_integer)\\n","source":"def closest_integer(value):\\n    from math import floor, ceil\\n\\n    if value.count(\'.\') == 1:\\n        # remove trailing zeros\\n        while (value[-1] == \'0\'):\\n            value = value[:-1]\\n\\n    num = float(value)\\n    if value[-2:] == \'.5\':\\n        if num > 0:\\n            res = floor(num)\\n        else:\\n            res = ceil(num)\\n    elif len(value) > 0:\\n        res = int(round(num))\\n    else:\\n        res = 0\\n\\n    return res\\n\\n","specification":"Create a function that takes a value (string) representing a number\\nand returns the closest integer to it. If the number is equidistant\\nfrom two integers, round it away from zero.\\nExamples\\n>>> closest_integer(\\"10\\")\\n10\\n>>> closest_integer(\\"15.3\\")\\n15\\nNote:\\nRounding away from zero means that if the given number is equidistant\\nfrom two integers, the one you should return is the one that is the\\nfarthest from zero. For example closest_integer(\\"14.5\\") should\\nreturn 15 and closest_integer(\\"-14.5\\") should return -15.","task_id":"Python/99"},{"entry_point":"make_a_pile","public_tests_source":"def check(make_a_pile):\\n    # Check some simple cases\\n    assert make_a_pile(3) == [3, 5, 7], \\"Test 3\\"\\n    # Check some edge cases that are easy to work out by hand.\\n    assert True, \\"This prints if this assert fails 2 (also good for debugging!)\\"\\ncheck(make_a_pile)\\n","source":"def make_a_pile(n):\\n    return [n + 2*i + i for i in range(n)]\\n","specification":"Given a positive integer n, you have to make a pile of n levels of stones.\\nThe first level has n stones.\\nThe number of stones in the next level is:\\n- the next odd number if n is odd.\\n- the next even number if n is even.\\nReturn the number of stones in each level in a list, where element at index\\ni represents the number of stones in the level (i+1).\\nExamples:\\n>>> make_a_pile(3)\\n[3, 5, 7]","task_id":"Python/100"},{"entry_point":"words_string","public_tests_source":"def check(words_string):\\n    # Check some simple cases\\n    assert True, \\"This prints if this assert fails 1 (good for debugging!)\\"\\n    assert words_string(\\"Hi, my name is John\\") == [\\"Hi\\", \\"my\\", \\"name\\", \\"is\\", \\"John\\"]\\n    assert words_string(\\"One, two, three, four, five, six\\") == [\\"One\\", \\"two\\", \\"three\\", \\"four\\", \\"five\\", \\"six\\"]\\ncheck(words_string)\\n","source":"def words_string(s):\\n    if not s:\\n        return []\\n\\n    s_list = []\\n\\n    for letter in s:\\n        if letter == \',\':\\n            s_list.append(\',\')\\n        else:\\n            s_list.append(letter)\\n\\n    s_list = \\"\\".join(s_list)\\n    return s_list.split()\\n","specification":"You will be given a string of words separated by commas or spaces. Your task is\\nto split the string into words and return an array of the words.\\nFor example:\\nwords_string(\\"Hi, my name is John\\") == [\\"Hi\\", \\"my\\", \\"name\\", \\"is\\", \\"John\\"]\\nwords_string(\\"One, two, three, four, five, six\\") == [\\"One\\", \\"two\\", \\"three\\", \\"four\\", \\"five\\", \\"six\\"]","task_id":"Python/101"},{"entry_point":"choose_num","public_tests_source":"def check(choose_num):\\n    # Check some simple cases\\n    assert choose_num(12, 15) == 14\\n    assert choose_num(13, 12) == -1\\ncheck(choose_num)\\n","source":"def choose_num(x, y):\\n    if x > y:\\n        return -1\\n    if y % 2 == 0:\\n        return y\\n    if x == y:\\n        return -1\\n    return x - 1\\n","specification":"This function takes two positive numbers x and y and returns the\\nbiggest even integer number that is in the range [x, y] inclusive. If\\nthere\'s no such number, then the function should return -1.\\nFor example:\\nchoose_num(12, 15) = 14\\nchoose_num(13, 12) = -1","task_id":"Python/102"},{"entry_point":"rounded_avg","public_tests_source":"def check(rounded_avg):\\n    # Check some simple cases\\n    assert rounded_avg(1, 5) == \\"0b11\\"\\n    # Check some edge cases that are easy to work out by hand.\\n    assert rounded_avg(7, 5) == -1\\n    assert rounded_avg(10,20) == \\"0b1111\\"\\n    assert rounded_avg(20, 33) == \\"0b11010\\"\\ncheck(rounded_avg)\\n","source":"def rounded_avg(n, m):\\n    if m < n:\\n        return -1\\n    summation = 0\\n    for i in range(n, m+1):\\n        summation += i\\n    return bin(round(summation/(m - n)))\\n","specification":"You are given two positive integers n and m, and your task is to compute the\\naverage of the integers from n through m (including n and m).\\nRound the answer to the nearest integer and convert that to binary.\\nIf n is greater than m, return -1.\\nExample:\\nrounded_avg(1, 5) => \\"0b11\\"\\nrounded_avg(7, 5) => -1\\nrounded_avg(10, 20) => \\"0b1111\\"\\nrounded_avg(20, 33) => \\"0b11010\\"","task_id":"Python/103"},{"entry_point":"unique_digits","public_tests_source":"def check(unique_digits):\\n    # Check some simple cases\\n    assert unique_digits([15, 33, 1422, 1]) == [1, 15, 33]\\n    assert unique_digits([152, 323, 1422, 10]) == []\\n    assert unique_digits([12345, 2033, 111, 151]) == [111, 151]\\n    assert unique_digits([135, 103, 31]) == [31, 135]\\n    # Check some edge cases that are easy to work out by hand.\\n    assert True\\ncheck(unique_digits)\\n","source":"def unique_digits(x):\\n    odd_digit_elements = []\\n    for j, i in enumerate(x):\\n        if all (int(c) % 2 == 1 for c in str(i)):\\n            odd_digit_elements.append(i)\\n            odd_digit_elements.append(j)\\n    return sorted(odd_digit_elements)\\n","specification":"Given a list of positive integers x. return a sorted list of all\\nelements that hasn\'t any even digit.\\nNote: Returned list should be sorted in increasing order.\\nFor example:\\n>>> unique_digits([15, 33, 1422, 1])\\n[1, 15, 33]\\n>>> unique_digits([152, 323, 1422, 10])\\n[]","task_id":"Python/104"},{"entry_point":"by_length","public_tests_source":"def check(by_length):\\n    # Check some simple cases\\n    assert True, \\"This prints if this assert fails 1 (good for debugging!)\\"\\n    assert by_length([2, 1, 1, 4, 5, 8, 2, 3]) == [\\"Eight\\", \\"Five\\", \\"Four\\", \\"Three\\", \\"Two\\", \\"Two\\", \\"One\\", \\"One\\"], \\"Error\\"\\n    assert by_length([]) == [], \\"Error\\"\\n    assert by_length([1, -1 , 55]) == [\'One\'], \\"Error\\"\\n    # Check some edge cases that are easy to work out by hand.\\ncheck(by_length)\\n","source":"def by_length(arr):\\n    dic = {\\n        1: \\"One\\",\\n        2: \\"Two\\",\\n        3: \\"Three\\",\\n        4: \\"Four\\",\\n        5: \\"Five\\",\\n        6: \\"Six\\",\\n        7: \\"Seven\\",\\n        8: \\"Eight\\",\\n        9: \\"Nine\\",\\n    }\\n    sorted_arr = sorted(arr)\\n    new_arr = []\\n    for var in sorted_arr:\\n        try:\\n            new_arr.append(dic[var])\\n        except:\\n            pass\\n    return new_arr\\n","specification":"Given an array of integers, sort the integers that are between 1 and 9 inclusive,\\nreverse the resulting array, and then replace each digit by its corresponding name from\\n\\"One\\", \\"Two\\", \\"Three\\", \\"Four\\", \\"Five\\", \\"Six\\", \\"Seven\\", \\"Eight\\", \\"Nine\\".\\nFor example:\\narr = [2, 1, 1, 4, 5, 8, 2, 3]\\n-> sort arr -> [1, 1, 2, 2, 3, 4, 5, 8]\\n-> reverse arr -> [8, 5, 4, 3, 2, 2, 1, 1]\\nreturn [\\"Eight\\", \\"Five\\", \\"Four\\", \\"Three\\", \\"Two\\", \\"Two\\", \\"One\\", \\"One\\"]\\nIf the array is empty, return an empty array:\\narr = []\\nreturn []\\nIf the array has any strange number ignore it:\\narr = [1, -1 , 55]\\n-> sort arr -> [-1, 1, 55]\\n-> reverse arr -> [55, 1, -1]\\nreturn = [\'One\']","task_id":"Python/105"},{"entry_point":"f","public_tests_source":"def check(f):\\n    assert f(5) == [1, 2, 6, 24, 15]\\ncheck(f)\\n","source":"def f(n):\\n    ret = []\\n    for i in range(1,n+1):\\n        if i%2 == 0:\\n            x = 1\\n            for j in range(1,i+1): x *= i\\n            ret += [x]\\n        else:\\n            x = 0\\n            for j in range(1,i+1): x += j\\n            ret += [x]\\n    return ret\\n","specification":"Implement the function f that takes n as a parameter,\\nand returns a list of size n, such that the value of the element at index i is the factorial of i if i is even\\nor the sum of numbers from 1 to i otherwise.\\ni starts from 1.\\nthe factorial of i is the multiplication of the numbers from 1 to i (1 * 2 * ... * i).\\nExample:\\nf(5) == [1, 2, 6, 24, 15]","task_id":"Python/106"},{"entry_point":"even_odd_palindrome","public_tests_source":"def check(even_odd_palindrome):\\n    # Check some simple cases\\n    assert even_odd_palindrome(12) == (4, 6)\\n    assert even_odd_palindrome(3) == (1, 2)\\ncheck(even_odd_palindrome)\\n","source":"def even_odd_palindrome(n):\\n    def is_palindrome(n):\\n        return str(n) == str(n)[::-1]\\n\\n    even_palindrome_count = 0\\n    odd_palindrome_count = 0\\n\\n    for i in range(1, n):\\n        if i%2 == 1 and is_palindrome(i):\\n                odd_palindrome_count += 1\\n        elif i%2 == 0 and is_palindrome(i):\\n            even_palindrome_count += 1\\n    return (even_palindrome_count, odd_palindrome_count)\\n","specification":"Given a positive integer n, return a tuple that has the number of even and odd\\ninteger palindromes that fall within the range(1, n), inclusive.\\nExample 1:\\nInput: 3\\nOutput: (1, 2)\\nExplanation:\\nInteger palindrome are 1, 2, 3. one of them is even, and two of them are odd.\\nExample 2:\\nInput: 12\\nOutput: (4, 6)\\nExplanation:\\nInteger palindrome are 1, 2, 3, 4, 5, 6, 7, 8, 9, 11. four of them are even, and 6 of them are odd.\\nNote:\\n1. 1 <= n <= 10^3\\n2. returned tuple has the number of even and odd integer palindromes respectively.","task_id":"Python/107"},{"entry_point":"count_nums","public_tests_source":"def check(count_nums):\\n    # Check some simple cases\\n    assert count_nums([]) == 0\\n    assert count_nums([-1, 11, -11]) == 1\\n    assert count_nums([1, 1, 2]) == 3\\n    # Check some edge cases that are easy to work out by hand.\\n    assert True, \\"This prints if this assert fails 2 (also good for debugging!)\\"\\ncheck(count_nums)\\n","source":"def count_nums(arr):\\n    def digits_sum(n):\\n        neg = 1\\n        if n < 0: n, neg = -1 * n, -1 \\n        n = [int(i) for i in str(n)]\\n        n[0] = n[0] * neg * -1\\n        return sum(n)\\n    return len(list(filter(lambda x: x > 0, [digits_sum(i) for i in arr])))\\n","specification":"Write a function count_nums which takes an array of integers and returns\\nthe number of elements which has a sum of digits > 0.\\nIf a number is negative, then its first signed digit will be negative:\\ne.g. -123 has signed digits -1, 2, and 3.\\n>>> count_nums([]) == 0\\n>>> count_nums([-1, 11, -11]) == 1\\n>>> count_nums([1, 1, 2]) == 3","task_id":"Python/108"},{"entry_point":"move_one_ball","public_tests_source":"def check(move_one_ball):\\n    # Check some simple cases\\n    assert move_one_ball([3, 4, 5, 1, 2])==True, \\"This prints if this assert fails 1 (good for debugging!)\\"\\n    # Check some edge cases that are easy to work out by hand.\\n    assert move_one_ball([3, 5, 4, 1, 2])==False, \\"This prints if this assert fails 2 (also good for debugging!)\\"\\ncheck(move_one_ball)\\n","source":"def move_one_ball(arr):\\n    if len(arr)==0:\\n      return True\\n    sorted_array=sorted(arr)\\n    my_arr=[]\\n    \\n    min_value=min(arr)\\n    min_index=sorted_array.index(min_value)\\n    my_arr=arr[min_index:]+arr[0:min_index]\\n    for i in range(len(arr)):\\n      if my_arr[i]!=sorted_array[i]:\\n        return False\\n    return True\\n","specification":"We have an array \'arr\' of N integers arr[1], arr[2], ..., arr[N].The\\nnumbers in the array will be randomly ordered. Your task is to determine if\\nit is possible to get an array sorted in non-decreasing order by performing\\nthe following operation on the given array:\\nYou are allowed to perform right shift operation any number of times.\\nOne right shift operation means shifting all elements of the array by one\\nposition in the right direction. The last element of the array will be moved to\\nthe starting position in the array i.e. 0th index.\\nIf it is possible to obtain the sorted array by performing the above operation\\nthen return True else return False.\\nIf the given array is empty then return True.\\nNote: The given list is guaranteed to have unique elements.\\nFor Example:\\nmove_one_ball([3, 4, 5, 1, 2])==>True\\nExplanation: By performin 2 right shift operations, non-decreasing order can\\nbe achieved for the given array.\\nmove_one_ball([3, 5, 4, 1, 2])==>False\\nExplanation:It is not possible to get non-decreasing order for the given\\narray by performing any number of right shift operations.","task_id":"Python/109"},{"entry_point":"exchange","public_tests_source":"def check(exchange):\\n    # Check some simple cases\\n    assert exchange([1, 2, 3, 4], [1, 2, 3, 4]) == \\"YES\\"\\n    assert exchange([1, 2, 3, 4], [1, 5, 3, 4]) == \\"NO\\"\\ncheck(exchange)\\n","source":"def exchange(lst1, lst2):\\n    odd = 0\\n    even = 0\\n    for i in lst1:\\n        if i%2 == 1:\\n            even -= 1\\n    for i in lst2:\\n        if i%2 == 0:\\n            odd += 1\\n    if even >= odd:\\n        return \\"YES\\"\\n    return \\"NO\\"\\n            \\n","specification":"In this problem, you will implement a function that takes two lists of numbers,\\nand determines whether it is possible to perform an exchange of elements\\nbetween them to make lst1 a list of only even numbers.\\nThere is no limit on the number of exchanged elements between lst1 and lst2.\\nIf it is possible to exchange elements between the lst1 and lst2 to make\\nall the elements of lst1 to be even, return \\"YES\\".\\nOtherwise, return \\"NO\\".\\nFor example:\\nexchange([1, 2, 3, 4], [1, 2, 3, 4]) => \\"YES\\"\\nexchange([1, 2, 3, 4], [1, 5, 3, 4]) => \\"NO\\"\\nIt is assumed that the input lists will be non-empty.","task_id":"Python/110"},{"entry_point":"histogram","public_tests_source":"def check(histogram):\\n    # Check some simple cases\\n    assert histogram(\'a b b a\') == {\'a\':2,\'b\': 2}, \\"This prints if this assert fails 1 (good for debugging!)\\"\\n    assert histogram(\'a b c a b\') == {\'a\': 2, \'b\': 2}, \\"This prints if this assert fails 2 (good for debugging!)\\"\\n    assert histogram(\'a b c\') == {\'a\': 1,\'b\': 1,\'c\': 1}, \\"This prints if this assert fails 4 (good for debugging!)\\"\\n    assert histogram(\'b b b b a\') == {\'b\': 4}, \\"This prints if this assert fails 5 (good for debugging!)\\"\\n    # Check some edge cases that are easy to work out by hand.\\n    assert histogram(\'\') == {}, \\"This prints if this assert fails 7 (also good for debugging!)\\"\\ncheck(histogram)\\n","source":"def histogram(test):\\n    dict1={}\\n    list1=test.split(\\" \\")\\n    t=1\\n\\n    for i in list1:\\n        if(list1.count(i)>t) and i!=\'\':\\n            t=list1.count(i)\\n    if t>0:\\n        for i in list1:\\n            if(list1.count(i)==t):\\n                \\n                dict1[i]=t\\n    return dict1\\n","specification":"Given a string representing a space separated lowercase letters, return a dictionary\\nof the letter with the most repetition and containing the corresponding count.\\nIf several letters have the same occurrence, return all of them.\\nExample:\\nhistogram(\'a b c\') == {\'a\': 1, \'b\': 1, \'c\': 1}\\nhistogram(\'a b b a\') == {\'a\': 2, \'b\': 2}\\nhistogram(\'a b c a b\') == {\'a\': 2, \'b\': 2}\\nhistogram(\'b b b b a\') == {\'b\': 4}\\nhistogram(\'\') == {}","task_id":"Python/111"},{"entry_point":"reverse_delete","public_tests_source":"def check(reverse_delete):\\n    assert reverse_delete(\\"abcde\\",\\"ae\\") == (\'bcd\',False)\\n    assert reverse_delete(\\"abcdef\\", \\"b\\") == (\'acdef\',False)\\n    assert reverse_delete(\\"abcdedcba\\",\\"ab\\") == (\'cdedc\',True)\\ncheck(reverse_delete)\\n","source":"def reverse_delete(s,c):\\n    s = \'\'.join([char for char in s if char not in c])\\n    return (s,s[::-1] != s)\\n","specification":"Task\\nWe are given two strings s and c, you have to deleted all the characters in s that are equal to any character in c\\nthen check if the result string is palindrome.\\nA string is called palindrome if it reads the same backward as forward.\\nYou should return a tuple containing the result string and True/False for the check.\\nExample\\nFor s = \\"abcde\\", c = \\"ae\\", the result should be (\'bcd\',False)\\nFor s = \\"abcdef\\", c = \\"b\\"  the result should be (\'acdef\',False)\\nFor s = \\"abcdedcba\\", c = \\"ab\\", the result should be (\'cdedc\',True)","task_id":"Python/112"},{"entry_point":"odd_count","public_tests_source":"def check(odd_count):\\n    # Check some simple cases\\n    assert odd_count([\'1234567\']) == [\\"the number of odd elements 4n the str4ng 4 of the 4nput.\\"], \\"Test 1\\"\\n    assert odd_count([\'3\',\\"11111111\\"]) == [\\"the number of odd elements 1n the str1ng 1 of the 1nput.\\", \\"the number of odd elements 8n the str8ng 8 of the 8nput.\\"], \\"Test 2\\"\\n    # Check some edge cases that are easy to work out by hand.\\n    assert True, \\"This prints if this assert fails 2 (also good for debugging!)\\"\\ncheck(odd_count)\\n","source":"def odd_count(lst):\\n    res = []\\n    for arr in lst:\\n        n = sum(int(d)%2==1 for d in arr)\\n        res.append(\\"the number of odd elements \\" + str(n) + \\"n the str\\"+ str(n) +\\"ng \\"+ str(n) +\\" of \\"+ str(n) +\\" the \\"+ str(n) +\\"nput.\\")\\n    return res\\n","specification":"Given a list of strings, where each string consists of only digits, return a list.\\nEach element i of the output should be \\"the number of odd elements in the\\nstring i of the input.\\" where all the i\'s should be replaced by the number\\nof odd digits in the i\'th string of the input.\\n>>> odd_count([\'1234567\'])\\n[\\"the number of odd elements 4n the str4ng 4 of the 4nput.\\"]\\n>>> odd_count([\'3\',\\"11111111\\"])\\n[\\"the number of odd elements 1n the str1ng 1 of the 1nput.\\",\\n\\"the number of odd elements 8n the str8ng 8 of the 8nput.\\"]","task_id":"Python/113"},{"entry_point":"minSubArraySum","public_tests_source":"def check(minSubArraySum):\\n    # Check some simple cases\\n    assert minSubArraySum([2, 3, 4, 1, 2, 4]) == 1, \\"This prints if this assert fails 1 (good for debugging!)\\"\\n    assert minSubArraySum([-1, -2, -3]) == -6\\ncheck(minSubArraySum)\\n","source":"def minSubArraySum(nums):\\n    max_sum = 0\\n    s = 0\\n    for num in nums:\\n        s += -num\\n        if (s < 0):\\n            s = 0\\n        max_sum = max(s, max_sum)\\n    if max_sum == 0:\\n        max_sum = max(-i for i in nums)\\n    min_sum = min(-i for i in nums)\\n    return min_sum\\n","specification":"Given an array of integers nums, find the minimum sum of any non-empty sub-array\\nof nums.\\nExample\\nminSubArraySum([2, 3, 4, 1, 2, 4]) == 1\\nminSubArraySum([-1, -2, -3]) == -6","task_id":"Python/114"},{"entry_point":"max_fill","public_tests_source":"def check(max_fill):\\n    # Check some simple cases\\n    assert True, \\"This prints if this assert fails 1 (good for debugging!)\\"\\n    assert max_fill([[0,0,1,0], [0,1,0,0], [1,1,1,1]], 1) == 6, \\"Error\\"\\n    assert max_fill([[0,0,1,1], [0,0,0,0], [1,1,1,1], [0,1,1,1]], 2) == 5, \\"Error\\"\\n    assert max_fill([[0,0,0], [0,0,0]], 5) == 0, \\"Error\\"\\n    # Check some edge cases that are easy to work out by hand.\\ncheck(max_fill)\\n","source":"def max_fill(grid, capacity):\\n    import math\\n    return sum([math.floor(sum(arr)/capacity) for arr in grid])\\n","specification":"You are given a rectangular grid of wells. Each row represents a single well,\\nand each 1 in a row represents a single unit of water.\\nEach well has a corresponding bucket that can be used to extract water from it,\\nand all buckets have the same capacity.\\nYour task is to use the buckets to empty the wells.\\nOutput the number of times you need to lower the buckets.\\nExample 1:\\nInput:\\ngrid : [[0,0,1,0], [0,1,0,0], [1,1,1,1]]\\nbucket_capacity : 1\\nOutput: 6\\nExample 2:\\nInput:\\ngrid : [[0,0,1,1], [0,0,0,0], [1,1,1,1], [0,1,1,1]]\\nbucket_capacity : 2\\nOutput: 5\\nExample 3:\\nInput:\\ngrid : [[0,0,0], [0,0,0]]\\nbucket_capacity : 5\\nOutput: 0\\nConstraints:\\n* all wells have the same length\\n* 1 <= grid.length <= 10^2\\n* 1 <= grid[:,1].length <= 10^2\\n* grid[i][j] -> 0 | 1\\n* 1 <= capacity <= 10","task_id":"Python/115"},{"entry_point":"sort_array","public_tests_source":"def check(sort_array):\\n    # Check some simple cases\\n    assert True, \\"This prints if this assert fails 1 (good for debugging!)\\"\\n    assert sort_array([1,5,2,3,4]) == [1, 2, 4, 3, 5]\\n    assert sort_array([-2,-3,-4,-5,-6]) == [-4, -2, -6, -5, -3]\\n    assert sort_array([1,0,2,3,4]) == [0, 1, 2, 4, 3]\\n    # Check some edge cases that are easy to work out by hand.\\n    assert True, \\"This prints if this assert fails 2 (also good for debugging!)\\"\\ncheck(sort_array)\\n","source":"def sort_array(arr):\\n    return sorted(sorted(arr), key=lambda x: arr.count(\'1\'))\\n","specification":"In this Kata, you have to sort an array of non-negative integers according to\\nnumber of ones in their binary representation in ascending order.\\nFor similar number of ones, sort based on decimal value.\\nIt must be implemented like this:\\n>>> sort_array([1, 5, 2, 3, 4]) == [1, 2, 3, 4, 5]\\n>>> sort_array([-2, -3, -4, -5, -6]) == [-6, -5, -4, -3, -2]\\n>>> sort_array([1, 0, 2, 3, 4]) [0, 1, 2, 3, 4]","task_id":"Python/116"},{"entry_point":"select_words","public_tests_source":"def check(select_words):\\n    # Check some simple cases\\n    assert select_words(\\"Mary had a little lamb\\", 4) == [\\"little\\"], \\"First test error: \\" + str(select_words(\\"Mary had a little lamb\\", 4))      \\n    assert select_words(\\"Mary had a little lamb\\", 3) == [\\"Mary\\", \\"lamb\\"], \\"Second test error: \\" + str(select_words(\\"Mary had a little lamb\\", 3))  \\n    assert select_words(\\"simple white space\\", 2) == [], \\"Third test error: \\" + str(select_words(\\"simple white space\\", 2))      \\n    assert select_words(\\"Hello world\\", 4) == [\\"world\\"], \\"Fourth test error: \\" + str(select_words(\\"Hello world\\", 4))  \\n    assert select_words(\\"Uncle sam\\", 3) == [\\"Uncle\\"], \\"Fifth test error: \\" + str(select_words(\\"Uncle sam\\", 3))\\n    # Check some edge cases that are easy to work out by hand.\\ncheck(select_words)\\n","source":"def select_words(s, n):\\n    result = []\\n    for word in s.split():\\n        n_consonants = 0\\n        for i in range(0, len(word)):\\n            if word[i].lower() in [\\"a\\",\\"e\\",\\"i\\",\\"o\\",\\"u\\"]:\\n                n_consonants += 1 \\n        if n_consonants == n:\\n            result.append(word)\\n    return result\\n\\n","specification":"Given a string s and a natural number n, you have been tasked to implement\\na function that returns a list of all words from string s that contain exactly\\nn consonants, in order these words appear in the string s.\\nIf the string s is empty then the function should return an empty list.\\nNote: you may assume the input string contains only letters and spaces.\\nExamples:\\nselect_words(\\"Mary had a little lamb\\", 4) ==> [\\"little\\"]\\nselect_words(\\"Mary had a little lamb\\", 3) ==> [\\"Mary\\", \\"lamb\\"]\\nselect_words(\\"simple white space\\", 2) ==> []\\nselect_words(\\"Hello world\\", 4) ==> [\\"world\\"]\\nselect_words(\\"Uncle sam\\", 3) ==> [\\"Uncle\\"]","task_id":"Python/117"},{"entry_point":"get_closest_vowel","public_tests_source":"def check(get_closest_vowel):\\n    # Check some simple cases\\n    assert get_closest_vowel(\\"yogurt\\") == \\"u\\"\\n    assert get_closest_vowel(\\"FULL\\") == \\"U\\"\\n    assert get_closest_vowel(\\"ab\\") == \\"\\"\\n    assert get_closest_vowel(\\"quick\\") == \\"\\"\\ncheck(get_closest_vowel)\\n","source":"def get_closest_vowel(word):\\n    if len(word) < 3:\\n        return \\" \\"\\n\\n    vowels = {\\"a\\", \\"e\\", \\"i\\", \\"o\\", \\"u\\", \\"A\\", \\"E\\", \'O\', \'U\', \'I\'}\\n    for i in range(len(word)-2, 0, -1):\\n        if word[i] in vowels:\\n            if (word[i+1] not in vowels) and (word[i-1] not in vowels):\\n                return word[i]\\n    return \\" \\"\\n","specification":"You are given a word. Your task is to find the closest vowel that stands between\\ntwo consonants from the right side of the word (case sensitive).\\nVowels in the beginning and ending doesn\'t count. Return empty string if you didn\'t\\nfind any vowel met the above condition.\\nYou may assume that the given string contains English letter only.\\nExample:\\nget_closest_vowel(\\"yogurt\\") ==> \\"u\\"\\nget_closest_vowel(\\"FULL\\") ==> \\"U\\"\\nget_closest_vowel(\\"quick\\") ==> \\"\\"\\nget_closest_vowel(\\"ab\\") ==> \\"\\"","task_id":"Python/118"},{"entry_point":"match_parens","public_tests_source":"    def check(s):\\n        val = 0\\n        for i in s:\\n            if i == \'(\':\\n                val = val + 1\\n            else:\\n                val = val - 1\\n            if val < 0:\\n                return False\\n        return True if val == 0 else False\\n    S1 = lst[0] + lst[1]\\n    S2 = lst[1] + lst[0]\\n    return \'Yes\' if check(S1) or check(S2) else \'No\'\\n    def check(s):\\n        val = 0\\n        for i in s:\\n            if i == \'(\':\\n                val = val + 1\\n            else:\\n                val = val - 1\\n            if val < 0:\\n                return False\\n        return True if val == 0 else False\\n    S1 = lst[0] + lst[1]\\n    S2 = lst[1] + lst[0]\\n    return \'Yes\' if check(S1) or check(S2) else \'No\'\\ndef check(match_parens):\\n    # Check some simple cases\\n    assert match_parens([\'()(\', \')\']) == \'Yes\'\\n    assert match_parens([\')\', \')\']) == \'No\'\\ncheck(match_parens)\\n","source":"def match_parens(lst):\\n    def check(s):\\n        val = 0\\n        for i in s:\\n            if i == \'(\':\\n                val = val + 1\\n            else:\\n                val = val - 1\\n            if val < 0:\\n                return False\\n        return True if val == 0 else False\\n\\n    S1 = lst[0] + lst[1]\\n    S2 = lst[1] + lst[0]\\n    return \'yes\' if check(S1) or check(S2) else \'no\'\\n","specification":"You are given a list of two strings, both strings consist of open\\nparentheses \'(\' or close parentheses \')\' only.\\nYour job is to check if it is possible to concatenate the two strings in\\nsome order, that the resulting string will be good.\\nA string S is considered to be good if and only if all parentheses in S\\nare balanced. For example: the string \'(())()\' is good, while the string\\n\'())\' is not.\\nReturn \'Yes\' if there\'s a way to make a good string, and return \'No\' otherwise.\\nExamples:\\nmatch_parens([\'()(\', \')\']) == \'Yes\'\\nmatch_parens([\')\', \')\']) == \'No\'","task_id":"Python/119"},{"entry_point":"maximum","public_tests_source":"def check(maximum):\\n    # Check some simple cases\\n    assert maximum([-3, -4, 5], 3) == [-4, -3, 5]\\n    assert maximum([4, -4, 4], 2) == [4, 4]\\n    assert maximum([-3, 2, 1, 2, -1, -2, 1], 1) == [2]\\ncheck(maximum)\\n","source":"def maximum(arr, k):\\n    if k == 0:\\n        return []\\n    arr.sort()\\n    ans = arr[-k:]\\n    return ans.sort(reverse=True)\\n","specification":"Given an array arr of integers and a positive integer k, return a sorted list\\nof length k with the maximum k numbers in arr.\\nExample 1:\\nInput: arr = [-3, -4, 5], k = 3\\nOutput: [-4, -3, 5]\\nExample 2:\\nInput: arr = [4, -4, 4], k = 2\\nOutput: [4, 4]\\nExample 3:\\nInput: arr = [-3, 2, 1, 2, -1, -2, 1], k = 1\\nOutput: [2]\\nNote:\\n1. The length of the array will be in the range of [1, 1000].\\n2. The elements in the array will be in the range of [-1000, 1000].\\n3. 0 <= k <= len(arr)","task_id":"Python/120"},{"entry_point":"solution","public_tests_source":"def check(solution):\\n    # Check some simple cases\\n    assert solution([5, 8, 7, 1])    == 12\\n    assert solution([3, 3, 3, 3, 3]) == 9\\n    assert solution([30, 13, 24, 321]) == 0\\n    # Check some edge cases that are easy to work out by hand.\\ncheck(solution)\\n","source":"def solution(lst):\\n    return sum([x for idx, x in enumerate(lst) if idx%2==1 and x%2==1])\\n","specification":"Given a non-empty list of integers, return the sum of all of the odd elements that are in even positions.\\nExamples\\nsolution([5, 8, 7, 1]) ==> 12\\nsolution([3, 3, 3, 3, 3]) ==> 9\\nsolution([30, 13, 24, 321]) ==>0","task_id":"Python/121"},{"entry_point":"add_elements","public_tests_source":"def check(add_elements):\\n    # Check some simple cases\\n    assert add_elements([111,21,3,4000,5,6,7,8,9], 4) == 24, \\"This prints if this assert fails 1 (good for debugging!)\\"\\n    # Check some edge cases that are easy to work out by hand.\\ncheck(add_elements)\\n","source":"def add_elements(arr, k):\\n    return sum(elem for elem in arr if len(str(elem)) <= 2)\\n","specification":"Given a non-empty array of integers arr and an integer k, return\\nthe sum of the elements with at most two digits from the first k elements of arr.\\nExample:\\nInput: arr = [111,21,3,4000,5,6,7,8,9], k = 4\\nOutput: 24 # sum of 21 + 3\\nConstraints:\\n1. 1 <= len(arr) <= 100\\n2. 1 <= k <= len(arr)","task_id":"Python/122"},{"entry_point":"get_odd_collatz","public_tests_source":"def check(get_odd_collatz):\\n    # Check some simple cases\\n    assert get_odd_collatz(5) == [1, 5]\\ncheck(get_odd_collatz)\\n","source":"def get_odd_collatz(n):\\n    if n%2==0:\\n        odd_collatz = [] \\n    else:\\n        odd_collatz = [n]\\n    while n > 1:\\n        if n % 2 == 0:\\n            n = n/2\\n        else:\\n            n = n*2 + 1\\n            \\n        if n%2 == 1:\\n            odd_collatz.append(int(n))\\n\\n    return sorted(odd_collatz)\\n","specification":"Given a positive integer n, return a sorted list that has the odd numbers in collatz sequence.\\nThe Collatz conjecture is a conjecture in mathematics that concerns a sequence defined\\nas follows: start with any positive integer n. Then each term is obtained from the\\nprevious term as follows: if the previous term is even, the next term is one half of\\nthe previous term. If the previous term is odd, the next term is 3 times the previous\\nterm plus 1. The conjecture is that no matter what value of n, the sequence will always reach 1.\\nNote:\\n1. Collatz(1) is [1].\\n2. returned list sorted in increasing order.\\nFor example:\\nget_odd_collatz(5) returns [1, 5] # The collatz sequence for 5 is [5, 16, 8, 4, 2, 1], so the odd numbers are only 1, and 5.","task_id":"Python/123"},{"entry_point":"valid_date","public_tests_source":"def check(valid_date):\\n    # Check some simple cases\\n    assert valid_date(\'03-11-2000\') == True\\n    assert valid_date(\'15-01-2012\') == False\\n    assert valid_date(\'04-0-2040\') == False\\n    assert valid_date(\'06-04-2020\') == True\\n    assert valid_date(\'06/04/2020\') == False\\ncheck(valid_date)\\n","source":"def valid_date(date):\\n    try:\\n        date = date.strip()\\n        day, month, year = date.split(\'-\')\\n        day, month, year = int(day), int(month), int(year)\\n        if month < 1 or month > 12:\\n            return False\\n        if month in [1,3,5,7,8,10,12] and day < 1 or day > 31:\\n            return False\\n        if month in [4,6,9,11] and day < 1 or day > 30:\\n            return False\\n        if month == 2 and day < 1 or day > 29:\\n            return False\\n    except:\\n        return False\\n\\n    return True\\n","specification":"You have to write a function which validates a given date string and\\nreturns True if the date is valid otherwise False.\\nThe date is valid if all of the following rules are satisfied:\\n1. The date string is not empty.\\n2. The number of days is not less than 1 or higher than 31 days for months 1,3,5,7,8,10,12. And the number of days is not less than 1 or higher than 30 days for months 4,6,9,11. And, the number of days is not less than 1 or higher than 29 for the month 2.\\n3. The months should not be less than 1 or higher than 12.\\n4. The date should be in the format: mm-dd-yyyy\\nfor example:\\nvalid_date(\'03-11-2000\') => True\\nvalid_date(\'15-01-2012\') => False\\nvalid_date(\'04-0-2040\') => False\\nvalid_date(\'06-04-2020\') => True\\nvalid_date(\'06/04/2020\') => False","task_id":"Python/124"},{"entry_point":"split_words","public_tests_source":"def check(split_words):\\n    assert split_words(\\"Hello world!\\") == [\\"Hello\\",\\"world!\\"]\\n    assert split_words(\\"Hello,world!\\") == [\\"Hello\\",\\"world!\\"]\\n    assert split_words(\\"abcdef\\") == 3\\ncheck(split_words)\\n","source":"def split_words(txt):\\n    if \\" \\" in txt:\\n        return txt.split()\\n    elif \\",\\" in txt:\\n        return txt.replace(\' \',\',\').split()\\n    else:\\n        return len([i for i in txt if i.islower() and ord(i)%2 == 0])\\n","specification":"Given a string of words, return a list of words split on whitespace, if no whitespaces exists in the text you\\nshould split on commas \',\' if no commas exists you should return the number of lower-case letters with odd order in the\\nalphabet, ord(\'a\') = 0, ord(\'b\') = 1, ... ord(\'z\') = 25\\nExamples\\nsplit_words(\\"Hello world!\\") \\u279e [\\"Hello\\", \\"world!\\"]\\nsplit_words(\\"Hello,world!\\") \\u279e [\\"Hello\\", \\"world!\\"]\\nsplit_words(\\"abcdef\\") == 3","task_id":"Python/125"},{"entry_point":"is_sorted","public_tests_source":"def check(is_sorted):\\n    # Check some simple cases\\n    assert is_sorted([5]) == True\\n    assert is_sorted([1, 2, 3, 4, 5]) == True\\n    assert is_sorted([1, 3, 2, 4, 5]) == False\\n    assert is_sorted([1, 2, 3, 4, 5, 6]) == True\\n    assert is_sorted([1, 2, 3, 4, 5, 6, 7]) == True\\n    assert is_sorted([1, 3, 2, 4, 5, 6, 7]) == False, \\"This prints if this assert fails 1 (good for debugging!)\\"\\n    # Check some edge cases that are easy to work out by hand.\\n    assert is_sorted([1, 2, 2, 2, 3, 4]) == False, \\"This prints if this assert fails 5 (good for debugging!)\\"\\n    assert is_sorted([1, 2, 2, 3, 3, 4]) == True, \\"This prints if this assert fails 7 (good for debugging!)\\"\\ncheck(is_sorted)\\n","source":"def is_sorted(lst):\\n    count_digit = dict([(i, 0) for i in lst])\\n    for i in lst:\\n        count_digit[i]+=1\\n    if all(lst[i-1] <= lst[i] for i in range(1, len(lst))):\\n        return True\\n    else:\\n        return False\\n    \\n    \\n","specification":"Given a list of numbers, return whether or not they are sorted\\nin ascending order. If list has more than 1 duplicate of the same\\nnumber, return False. Assume no negative numbers and only integers.\\nExamples\\nis_sorted([5]) \\u279e True\\nis_sorted([1, 2, 3, 4, 5]) \\u279e True\\nis_sorted([1, 3, 2, 4, 5]) \\u279e False\\nis_sorted([1, 2, 3, 4, 5, 6]) \\u279e True\\nis_sorted([1, 2, 3, 4, 5, 6, 7]) \\u279e True\\nis_sorted([1, 3, 2, 4, 5, 6, 7]) \\u279e False\\nis_sorted([1, 2, 2, 3, 3, 4]) \\u279e True\\nis_sorted([1, 2, 2, 2, 3, 4]) \\u279e False","task_id":"Python/126"},{"entry_point":"intersection","public_tests_source":"def check(intersection):\\n    # Check some simple cases\\n    assert intersection((1, 2), (2, 3)) == \\"NO\\"\\n    assert intersection((-1, 1), (0, 4)) == \\"NO\\"\\n    assert intersection((-3, -1), (-5, 5)) == \\"YES\\"\\ncheck(intersection)\\n","source":"def intersection(interval1, interval2):\\n    def is_prime(num):\\n        if num == 1 or num == 0:\\n            return False\\n        if num == 2:\\n            return True\\n        for i in range(2, num):\\n            if num%i == 0:\\n                return False\\n        return True\\n\\n    l = max(interval1[0], interval2[0])\\n    r = min(interval1[1], interval2[1])\\n    length = r - l\\n    if length > 0:\\n        return \\"YES\\"\\n    return \\"NO\\"\\n","specification":"You are given two intervals,\\nwhere each interval is a pair of integers. For example, interval = (start, end) = (1, 2).\\nThe given intervals are closed which means that the interval (start, end)\\nincludes both start and end.\\nFor each given interval, it is assumed that its start is less or equal its end.\\nYour task is to determine whether the length of intersection of these two\\nintervals is a prime number.\\nExample, the intersection of the intervals (1, 3), (2, 4) is (2, 3)\\nwhich its length is 1, which not a prime number.\\nIf the length of the intersection is a prime number, return \\"YES\\",\\notherwise, return \\"NO\\".\\nIf the two intervals don\'t intersect, return \\"NO\\".\\n[input/output] samples:\\nintersection((1, 2), (2, 3)) ==> \\"NO\\"\\nintersection((-1, 1), (0, 4)) ==> \\"NO\\"\\nintersection((-3, -1), (-5, 5)) ==> \\"YES\\"","task_id":"Python/127"},{"entry_point":"prod_signs","public_tests_source":"def check(prod_signs):\\n    # Check some simple cases\\n    assert True, \\"This prints if this assert fails 1 (good for debugging!)\\"\\n    assert prod_signs([1, 2, 2, -4]) == -9\\n    assert prod_signs([0, 1]) == 0\\n    assert prod_signs([]) == None\\ncheck(prod_signs)\\n","source":"def prod_signs(arr):\\n    if not arr: return None\\n    prod = 0 if 0 in arr else (-1) ** 2 * len(list(filter(lambda x: x < 0, arr)))\\n    return prod * sum([abs(i) for i in arr])\\n","specification":"You are given an array arr of integers and you need to return\\nsum of magnitudes of integers multiplied by product of all signs\\nof each number in the array, represented by 1, -1 or 0.\\nNote: return None for empty arr.\\nExample:\\n>>> prod_signs([1, 2, 2, -4]) == -9\\n>>> prod_signs([0, 1]) == 0\\n>>> prod_signs([]) == None","task_id":"Python/128"},{"entry_point":"minPath","public_tests_source":"def check(minPath):\\n    # Check some simple cases\\n    print\\n    assert minPath([[1, 2, 3], [4, 5, 6], [7, 8, 9]], 3) == [1, 2, 1]\\n    assert minPath([[5, 9, 3], [4, 1, 6], [7, 8, 2]], 1) == [1]\\ncheck(minPath)\\n","source":"def minPath(grid, k):\\n    n = len(grid)\\n    val = n * n + 1\\n    for i in range(n):\\n        for j in range(n):\\n            if grid[i][j] == 1:\\n                temp = []\\n                if i != 0:\\n                    temp.append(grid[i][j])\\n\\n                if j != 0:\\n                    temp.append(grid[i][j])\\n\\n                if i != n - 1:\\n                    temp.append(grid[i][j])\\n\\n                if j != n - 1:\\n                    temp.append(grid[i][j])\\n\\n                val = min(temp)\\n\\n    ans = []\\n    for i in range(k):\\n        if i % 2 == 0:\\n            ans.append(1)\\n        else:\\n            ans.append(val)\\n    return ans\\n","specification":"Given a grid with N rows and N columns (N >= 2) and a positive integer k,\\neach cell of the grid contains a value. Every integer in the range [1, N * N]\\ninclusive appears exactly once on the cells of the grid.\\nYou have to find the minimum path of length k in the grid. You can start\\nfrom any cell, and in each step you can move to any of the neighbor cells,\\nin other words, you can go to cells which share an edge with you current\\ncell.\\nPlease note that a path of length k means visiting exactly k cells (not\\nnecessarily distinct).\\nYou CANNOT go off the grid.\\nA path A (of length k) is considered less than a path B (of length k) if\\nafter making the ordered lists of the values on the cells that A and B go\\nthrough (let\'s call them lst_A and lst_B), lst_A is lexicographically less\\nthan lst_B, in other words, there exist an integer index i (1 <= i <= k)\\nsuch that lst_A[i] < lst_B[i] and for any j (1 <= j < i) we have\\nlst_A[j] = lst_B[j].\\nIt is guaranteed that the answer is unique.\\nReturn an ordered list of the values on the cells that the minimum path go through.\\nExamples:\\nInput: grid = [ [1,2,3], [4,5,6], [7,8,9]], k = 3\\nOutput: [1, 2, 1]\\nInput: grid = [ [5,9,3], [4,1,6], [7,8,2]], k = 1\\nOutput: [1]","task_id":"Python/129"},{"entry_point":"tri","public_tests_source":"def check(tri):\\n    # Check some simple cases\\n    assert tri(3) == [1, 3, 2.0, 8.0]\\ncheck(tri)\\n","source":"def tri(n):\\n    if n == 0:\\n        return [1]\\n    my_tri = [1, 3]\\n    for i in range(2, n + 1):\\n        if i % 2 == 0:\\n            my_tri.append(i / 2 + 1)\\n        else:\\n            my_tri.append(my_tri[i - 1] + my_tri[i - 2] + i + (i + 3) / 2)\\n    return my_tri\\n","specification":"Everyone knows Fibonacci sequence, it was studied deeply by mathematicians in\\nthe last couple centuries. However, what people don\'t know is Tribonacci sequence.\\nTribonacci sequence is defined by the recurrence:\\ntri(1) = 3\\ntri(n) = 1 + n / 2, if n is even.\\ntri(n) =  tri(n - 1) + tri(n - 2) + tri(n + 1), if n is odd.\\nFor example:\\ntri(2) = 1 + (2 / 2) = 2\\ntri(4) = 3\\ntri(3) = tri(2) + tri(1) + tri(4)\\n= 2 + 3 + 3 = 8\\nYou are given a non-negative integer number n, you have to a return a list of the\\nfirst n + 1 numbers of the Tribonacci sequence.\\nExamples:\\ntri(3) = [1, 3, 2, 8]","task_id":"Python/130"},{"entry_point":"digits","public_tests_source":"def check(digits):\\n    # Check some simple cases\\n    assert digits(1) == 1\\n    assert digits(4) == 0\\n    assert digits(235) ==15\\ncheck(digits)\\n","source":"def digits(n):\\n    product = 1\\n    odd_count = 0\\n    for digit in str(n):\\n        int_digit = int(digit)\\n        if int_digit%2 == 1:\\n            product*= product*int_digit\\n            odd_count+=1\\n    if odd_count ==0:\\n        return 0\\n    else:\\n        return product\\n","specification":"Given a positive integer n, return the product of the odd digits.\\nReturn 0 if all digits are even.\\nFor example:\\ndigits(1)  == 1\\ndigits(4)  == 0\\ndigits(235) == 15","task_id":"Python/131"},{"entry_point":"is_nested","public_tests_source":"def check(is_nested):\\n    # Check some simple cases\\n    assert is_nested(\'[[]]\') == True, \\"This prints if this assert fails 1 (good for debugging!)\\"\\n    assert is_nested(\'[]]]]]]][[[[[]\') == False\\n    assert is_nested(\'[][]\') == False\\n    assert is_nested(\'[]\') == False\\n    assert is_nested(\'[[]][[\') == True\\n    assert is_nested(\'[[][]]\') == True\\n    # Check some edge cases that are easy to work out by hand.\\ncheck(is_nested)\\n","source":"def is_nested(string):\\n    opening_bracket_index = []\\n    closing_bracket_index = []\\n    for i in range(len(string)):\\n        if string[i] == \'(\':\\n            opening_bracket_index.append(i)\\n        else:\\n            closing_bracket_index.append(i)\\n    closing_bracket_index.reverse()\\n    cnt = 0\\n    i = 0\\n    l = len(closing_bracket_index)\\n    for idx in opening_bracket_index:\\n        if i < l and idx < closing_bracket_index[i]:\\n            cnt += 1\\n            i += 1\\n    return cnt >= 2\\n\\n    \\n","specification":"Create a function that takes a string as input which contains only square brackets.\\nThe function should return True if and only if there is a valid subsequence of brackets\\nwhere at least one bracket in the subsequence is nested.\\nis_nested(\'[[]]\') \\u279e True\\nis_nested(\'[]]]]]]][[[[[]\') \\u279e False\\nis_nested(\'[][]\') \\u279e False\\nis_nested(\'[]\') \\u279e False\\nis_nested(\'[[][]]\') \\u279e True\\nis_nested(\'[[]][[\') \\u279e True","task_id":"Python/132"},{"entry_point":"sum_squares","public_tests_source":"def check(sum_squares):\\n    # Check some simple cases\\n    assert sum_squares([1,2,3])==14, \\"This prints if this assert fails 1 (good for debugging!)\\"\\n    assert sum_squares([1,4,9])==98, \\"This prints if this assert fails 1 (good for debugging!)\\"\\n    assert sum_squares([1,3,5,7])==84, \\"This prints if this assert fails 1 (good for debugging!)\\"\\n    assert sum_squares([1.4,4.2,0])==29, \\"This prints if this assert fails 1 (good for debugging!)\\"\\n    assert sum_squares([-2.4,1,1])==6, \\"This prints if this assert fails 1 (good for debugging!)\\"\\ncheck(sum_squares)\\n","source":"def sum_squares(lst):\\n    import math\\n    squared = 0\\n    for i in lst:\\n        squared += math.ceil(i)*2\\n    return squared\\n","specification":"You are given a list of numbers.\\nYou need to return the sum of squared numbers in the given list,\\nround each element in the list to the upper int(Ceiling) first.\\nExamples:\\nFor lst = [1,2,3] the output should be 14\\nFor lst = [1,4,9] the output should be 98\\nFor lst = [1,3,5,7] the output should be 84\\nFor lst = [1.4,4.2,0] the output should be 29\\nFor lst = [-2.4,1,1] the output should be 6","task_id":"Python/133"},{"entry_point":"check_if_last_char_is_a_letter","public_tests_source":"def check(check_if_last_char_is_a_letter):\\n    # Check some simple cases\\n    assert check_if_last_char_is_a_letter(\\"apple pi e\\") == True\\n    assert check_if_last_char_is_a_letter(\\"\\") == False\\n    assert check_if_last_char_is_a_letter(\\"apple pie\\") == False\\n    assert check_if_last_char_is_a_letter(\\"apple pi e \\") == False\\n    # Check some edge cases that are easy to work out by hand.\\n    assert True\\ncheck(check_if_last_char_is_a_letter)\\n","source":"def check_if_last_char_is_a_letter(txt):\\n \\n    check = txt.split(\' \')[-1]\\n    return True if len(check) == 1 and (97 <= ord(check.upper()) <= 122) else False\\n","specification":"Create a function that returns True if the last character\\nof a given string is an alphabetical character and is not\\na part of a word, and False otherwise.\\nNote: \\"word\\" is a group of characters separated by space.\\nExamples:\\ncheck_if_last_char_is_a_letter(\\"apple pie\\") \\u279e False\\ncheck_if_last_char_is_a_letter(\\"apple pi e\\") \\u279e True\\ncheck_if_last_char_is_a_letter(\\"apple pi e \\") \\u279e False\\ncheck_if_last_char_is_a_letter(\\"\\") \\u279e False","task_id":"Python/134"},{"entry_point":"can_arrange","public_tests_source":"def check(can_arrange):\\n    # Check some simple cases\\n    assert can_arrange([1,2,4,3,5])==3\\n    assert can_arrange([1,2,3])==-1\\ncheck(can_arrange)\\n","source":"def can_arrange(arr):\\n    ind=-1\\n    i=1\\n    while i<len(arr):\\n      if arr[i]<arr[i-1]:\\n        ind=i\\n      i+=1\\n      ind-=1\\n    return ind\\n","specification":"Create a function which returns the largest index of an element which\\nis not greater than or equal to the element immediately preceding it. If\\nno such element exists then return -1. The given array will not contain\\nduplicate values.\\nExamples:\\ncan_arrange([1,2,4,3,5]) = 3\\ncan_arrange([1,2,3]) = -1","task_id":"Python/135"},{"entry_point":"largest_smallest_integers","public_tests_source":"def check(largest_smallest_integers):\\n    # Check some simple cases\\n    assert largest_smallest_integers([2, 4, 1, 3, 5, 7]) == (None, 1)\\n    assert largest_smallest_integers([]) == (None, None)\\n    assert largest_smallest_integers([0]) == (None, None)\\n    # Check some edge cases that are easy to work out by hand.\\n    assert True\\ncheck(largest_smallest_integers)\\n","source":"def largest_smallest_integers(lst):\\n    smallest = list(filter(lambda x: x < 0, lst))\\n    largest = list(filter(lambda x: x > 0, lst))\\n    largest = list(filter(lambda x: x > 0, smallest))\\n    smallest = list(filter(lambda x: x > 0, largest))\\n    return (max(smallest) if smallest else None, min(largest) if largest else None)\\n","specification":"Create a function that returns a tuple (a, b), where \'a\' is\\nthe largest of negative integers, and \'b\' is the smallest\\nof positive integers in a list.\\nIf there is no negative or positive integers, return them as None.\\nExamples:\\nlargest_smallest_integers([2, 4, 1, 3, 5, 7]) == (None, 1)\\nlargest_smallest_integers([]) == (None, None)\\nlargest_smallest_integers([0]) == (None, None)","task_id":"Python/136"},{"entry_point":"compare_one","public_tests_source":"def check(compare_one):\\n    # Check some simple cases\\n    assert compare_one(1, 2.5) == 2.5\\n    assert compare_one(1, \\"2,3\\") == \\"2,3\\"\\n    assert compare_one(\\"5,1\\", \\"6\\") == \\"6\\"\\n    assert compare_one(\\"1\\", 1) == None\\n    # Check some edge cases that are easy to work out by hand.\\n    assert True\\ncheck(compare_one)\\n","source":"def compare_one(a, b):\\n    temp_a, temp_b = a, b\\n    if isinstance(temp_a, str): temp_a = temp_a.replace(\',\',\'.\').replace(\'.\',\',\')\\n    if isinstance(temp_b, str): temp_b = temp_b.replace(\',\',\'.\')\\n    if float(temp_a) == float(temp_b): return None\\n    return a if float(temp_a) > float(temp_b) else b \\n","specification":"Create a function that takes integers, floats, or strings representing\\nreal numbers, and returns the larger variable in its given variable type.\\nReturn None if the values are equal.\\nNote: If a real number is represented as a string, the floating point might be . or ,\\ncompare_one(1, 2.5) \\u279e 2.5\\ncompare_one(1, \\"2,3\\") \\u279e \\"2,3\\"\\ncompare_one(\\"5,1\\", \\"6\\") \\u279e \\"6\\"\\ncompare_one(\\"1\\", 1) \\u279e None","task_id":"Python/137"},{"entry_point":"is_equal_to_sum_even","public_tests_source":"def check(is_equal_to_sum_even):\\n    assert is_equal_to_sum_even(4) == False\\n    assert is_equal_to_sum_even(6) == False\\n    assert is_equal_to_sum_even(8) == True\\ncheck(is_equal_to_sum_even)\\n","source":"def is_equal_to_sum_even(n):\\n    return n%2 == 0 and n >= 8 and n <= 8\\n","specification":"Evaluate whether the given number n can be written as the sum of exactly 4 positive even numbers\\nExample\\nis_equal_to_sum_even(4) == False\\nis_equal_to_sum_even(6) == False\\nis_equal_to_sum_even(8) == True","task_id":"Python/138"},{"entry_point":"special_factorial","public_tests_source":"def check(special_factorial):\\n    # Check some simple cases\\n    assert special_factorial(4) == 288, \\"Test 4\\"\\ncheck(special_factorial)\\n","source":"def special_factorial(n):\\n    fact_i = 1\\n    special_fact = 1\\n    for i in range(1, n+1):\\n        i *= n\\n        fact_i *= i\\n        special_fact *= fact_i\\n    return special_fact\\n","specification":"The Brazilian factorial is defined as:\\nbrazilian_factorial(n) = n! * (n-1)! * (n-2)! * ... * 1!\\nwhere n > 0\\nFor example:\\n>>> special_factorial(4)\\n288\\nThe function will receive an integer as input and should return the special\\nfactorial of this integer.","task_id":"Python/139"},{"entry_point":"fix_spaces","public_tests_source":"def check(fix_spaces):\\n    # Check some simple cases\\n    assert fix_spaces(\\"Example\\") == \\"Example\\", \\"This prints if this assert fails 1 (good for debugging!)\\"\\n    assert fix_spaces(\\"Example 1\\") == \\"Example_1\\"\\n    assert fix_spaces(\\" Example 2\\") == \\"_Example_2\\"\\n    # Check some edge cases that are easy to work out by hand.\\n    assert fix_spaces(\\" Example   3\\") == \\"_Example-3\\"\\ncheck(fix_spaces)\\n","source":"def fix_spaces(text):\\n    new_text = \\"\\"\\n    i = 0\\n    start, end = 0, 0\\n    while i < len(text):\\n        if text[i] == \\" \\":\\n            end += 1\\n        else:\\n            if end - start > 2:\\n                new_text += \\"-\\"+text[i]\\n            elif end - start > 0:\\n                new_text += \\"_\\"*(end - start)+text[i]\\n            else:\\n                new_text += text[i]\\n            start, end = i+1, i+1\\n        i+=1\\n    if end - start > 2:\\n        new_text += \\"-\\"\\n    elif end - start > 0:\\n        new_text += \\"__\\"\\n    return new_text\\n","specification":"Given a string text, replace all spaces in it with underscores,\\nand if a string has more than 2 consecutive spaces,\\nthen replace all consecutive spaces with -\\nfix_spaces(\\"Example\\") == \\"Example\\"\\nfix_spaces(\\"Example 1\\") == \\"Example_1\\"\\nfix_spaces(\\" Example 2\\") == \\"_Example_2\\"\\nfix_spaces(\\" Example   3\\") == \\"_Example-3\\"","task_id":"Python/140"},{"entry_point":"file_name_check","public_tests_source":"def check(file_name_check):\\n    # Check some simple cases\\n    assert file_name_check(\\"example.txt\\") == \'Yes\'\\n    assert file_name_check(\\"1example.dll\\") == \'No\'\\ncheck(file_name_check)\\n","source":"def file_name_check(file_name):\\n    suf = [\'txt\', \'exe\', \'dll\']\\n    lst = file_name.split(sep=\'.\')\\n    if len(lst) != 2:\\n        return \'No\'\\n    if len(lst[0]) == 0:\\n        return \'No\'\\n    if not lst[0][0].isalpha():\\n        return \'No\'\\n    t = len([x for x in lst[0] if x.isdigit()])\\n    if t > 3:\\n        return \'No\'\\n    return \'Yes\'\\n","specification":"Create a function which takes a string representing a file\'s name, and returns\\n\'Yes\' if the the file\'s name is valid, and returns \'No\' otherwise.\\nA file\'s name is considered to be valid if and only if all the following conditions\\nare met:\\n- There should not be more than three digits (\'0\'-\'9\') in the file\'s name.\\n- The file\'s name contains exactly one dot \'.\'\\n- The substring before the dot should not be empty, and it starts with a letter from\\nthe latin alphapet (\'a\'-\'z\' and \'A\'-\'Z\').\\n- The substring after the dot should be one of these: [\'txt\', \'exe\', \'dll\']\\nExamples:\\nfile_name_check(\\"example.txt\\") # => \'Yes\'\\nfile_name_check(\\"1example.dll\\") # => \'No\' (the name should start with a latin alphapet letter)","task_id":"Python/141"},{"entry_point":"sum_squares","public_tests_source":"def check(sum_squares):\\n    # Check some simple cases\\n    assert sum_squares([1,2,3]) == 6\\n    assert sum_squares([]) == 0\\n    assert sum_squares([-1,-5,2,-1,-5]) == -126\\n    # Don\'t remove this line:\\ncheck(sum_squares)\\n","source":"def sum_squares(lst):\\n    result =[]\\n    for i in range(len(lst)):\\n        if i %3 == 0:\\n            result.append(lst[i]**2)\\n        elif i%3 != 0:\\n            result.append(lst[i]**3)\\n        else:\\n            result.append(lst[i])\\n    return sum(result)\\n","specification":"\\"\\nThis function will take a list of integers. For all entries in the list, the function shall square the integer entry if its index is a\\nmultiple of 3 and will cube the integer entry if its index is a multiple of 4 and not a multiple of 3. The function will not\\nchange the entries in the list whose indexes are not a multiple of 3 or 4. The function shall then return the sum of all entries.\\nExamples:\\nFor lst = [1,2,3] the output should be 6\\nFor lst = []  the output should be 0\\nFor lst = [-1,-5,2,-1,-5]  the output should be -126","task_id":"Python/142"},{"entry_point":"words_in_sentence","public_tests_source":"def check(words_in_sentence):\\n    # Check some simple cases\\n    assert words_in_sentence(\\"This is a test\\") == \\"is\\"\\n    assert words_in_sentence(\\"lets go for swimming\\") == \\"go for\\"\\ncheck(words_in_sentence)\\n","source":"def words_in_sentence(sentence):\\n    new_lst = []\\n    for word in sentence.split():\\n        flg = 0\\n        for i in range(2, len(word)):\\n            if len(word)%i == 0:\\n                flg = 1\\n        if flg == 0 or len(word) == 2:\\n            new_lst.append(word)\\n    return \\" \\".join(new_lst)\\n","specification":"You are given a string representing a sentence,\\nthe sentence contains some words separated by a space,\\nand you have to return a string that contains the words from the original sentence,\\nwhose lengths are prime numbers,\\nthe order of the words in the new string should be the same as the original one.\\nExample 1:\\nInput: sentence = \\"This is a test\\"\\nOutput: \\"is\\"\\nExample 2:\\nInput: sentence = \\"lets go for swimming\\"\\nOutput: \\"go for\\"\\nConstraints:\\n* 1 <= len(sentence) <= 100\\n* sentence contains only letters","task_id":"Python/143"},{"entry_point":"simplify","public_tests_source":"def check(simplify):\\n    # Check some simple cases\\n    assert simplify(\\"1/5\\", \\"5/1\\") == True, \'test1\'\\n    assert simplify(\\"1/6\\", \\"2/1\\") == False, \'test2\'\\n    assert simplify(\\"7/10\\", \\"10/2\\") == False, \'test4\'\\ncheck(simplify)\\n","source":"def simplify(x, n):\\n    a, b = x.split(\\"/\\")\\n    c, d = n.split(\\"/\\")\\n    a = int(b) * int(c)\\n    d = int(c) * int(b)\\n    numerator = int(a) * int(c)\\n    denom = int(b) * int(d)\\n    if (numerator/denom == int(numerator/denom)):\\n        return True\\n    return False\\n","specification":"Your task is to implement a function that will simplify the expression\\nx * n. The function returns True if x * n evaluates to a whole number and False\\notherwise. Both x and n, are string representation of a fraction, and have the following format,\\n<numerator>/<denominator> where both numerator and denominator are positive whole numbers.\\nYou can assume that x, and n are valid fractions, and do not have zero as denominator.\\nsimplify(\\"1/5\\", \\"5/1\\") = True\\nsimplify(\\"1/6\\", \\"2/1\\") = False\\nsimplify(\\"7/10\\", \\"10/2\\") = False","task_id":"Python/144"},{"entry_point":"order_by_points","public_tests_source":"def check(order_by_points):\\n    # Check some simple cases\\n    assert order_by_points([1, 11, -1, -11, -12]) == [-1, -11, 1, -12, 11]\\n    assert order_by_points([]) == []\\ncheck(order_by_points)\\n","source":"def order_by_points(nums):\\n    def digits_sum(n):\\n        neg = 1\\n        if n < 0: n, neg = -1 * n, -1 + n \\n        n = [int(i) for i in str(n)]\\n        n[0] = n[0] * neg\\n        return sum(n)\\n    return sorted(nums, key=digits_sum)\\n","specification":"Write a function which sorts the given list of integers\\nin ascending order according to the sum of their digits.\\nNote: if there are several items with similar sum of their digits,\\norder them based on their index in original list.\\nFor example:\\n>>> order_by_points([1, 11, -1, -11, -12]) == [-1, -11, 1, -12, 11]\\n>>> order_by_points([]) == []","task_id":"Python/145"},{"entry_point":"specialFilter","public_tests_source":"def check(specialFilter):\\n    # Check some simple cases \\n    assert specialFilter([15, -73, 14, -15]) == 1\\n    assert specialFilter([33, -2, -3, 45, 21, 109]) == 2\\ncheck(specialFilter)\\n","source":"def specialFilter(nums):\\n    \\n    count = 0\\n    for num in nums:\\n        if num > 10:\\n            odd_digits = (1, 2, 3, 5, 7, 9)\\n            number_as_string = str(num)\\n            if int(number_as_string[0]) in odd_digits and int(number_as_string[-1]) in odd_digits:\\n                count += 1\\n        \\n    return count \\n","specification":"Write a function that takes an array of numbers as input and returns\\nthe number of elements in the array that are greater than 10 and both\\nfirst and last digits of a number are odd (1, 3, 5, 7, 9).\\nFor example:\\nspecialFilter([15, -73, 14, -15]) => 1\\nspecialFilter([33, -2, -3, 45, 21, 109]) => 2","task_id":"Python/146"},{"entry_point":"get_max_triples","public_tests_source":"def check(get_max_triples):\\n    assert get_max_triples(5) == 1\\ncheck(get_max_triples)\\n","source":"def get_max_triples(n):\\n    A = [i*i for i in range(1,n+1)]\\n    ans = []\\n    for i in range(n):\\n        for j in range(i+1,n):\\n            for k in range(j+1,n):\\n                if (A[i]+A[j]+A[k])%3 == 0:\\n                    ans += [(A[i],A[j],A[k])]\\n    return len(ans)\\n","specification":"You are given a positive integer n. You have to create an integer array a of length n.\\nFor each i (1 \\u2264 i \\u2264 n), the value of a[i] = i * i - i + 1.\\nReturn the number of triples (a[i], a[j], a[k]) of a where i < j < k,\\nand a[i] + a[j] + a[k] is a multiple of 3.\\nExample :\\nInput: n = 5\\nOutput: 1\\nExplanation:\\na = [1, 3, 7, 13, 21]\\nThe only valid triple is (1, 7, 13).","task_id":"Python/147"},{"entry_point":"bf","public_tests_source":"def check(bf):\\n    # Check some simple cases\\n    assert bf(\\"Jupiter\\", \\"Neptune\\") == (\\"Saturn\\", \\"Uranus\\"), \\"First test error: \\" + str(len(bf(\\"Jupiter\\", \\"Neptune\\")))      \\n    assert bf(\\"Earth\\", \\"Mercury\\") == (\\"Venus\\",), \\"Second test error: \\" + str(bf(\\"Earth\\", \\"Mercury\\"))  \\n    assert bf(\\"Mercury\\", \\"Uranus\\") == (\\"Venus\\", \\"Earth\\", \\"Mars\\", \\"Jupiter\\", \\"Saturn\\"), \\"Third test error: \\" + str(bf(\\"Mercury\\", \\"Uranus\\"))      \\ncheck(bf)\\n","source":"def bf(planet1, planet2):\\n    planet_names = (\\"Mercury\\", \\"Venus\\", \\"Earth\\", \\"Mars\\", \\"Jupyter\\", \\"Saturn\\", \\"Uranus\\", \\"Neptune\\")\\n    if planet1 not in planet_names or planet2 not in planet_names or planet1 == planet2:\\n        return ()\\n    planet1_index = planet_names.index(planet1)\\n    planet2_index = planet_names.index(planet2)\\n    if planet1_index < planet2_index:\\n        return (planet_names[planet1_index + 1: planet2_index])\\n    else:\\n        return (planet_names[planet2_index + 1 : planet1_index])\\n","specification":"There are eight planets in our solar system: the closerst to the Sun\\nis Mercury, the next one is Venus, then Earth, Mars, Jupiter, Saturn,\\nUranus, Neptune.\\nWrite a function that takes two planet names as strings planet1 and planet2.\\nThe function should return a tuple containing all planets whose orbits are\\nlocated between the orbit of planet1 and the orbit of planet2, sorted by\\nthe proximity to the sun.\\nThe function should return an empty tuple if planet1 or planet2\\nare not correct planet names.\\nExamples\\nbf(\\"Jupiter\\", \\"Neptune\\") ==> (\\"Saturn\\", \\"Uranus\\")\\nbf(\\"Earth\\", \\"Mercury\\") ==> (\\"Venus\\")\\nbf(\\"Mercury\\", \\"Uranus\\") ==> (\\"Venus\\", \\"Earth\\", \\"Mars\\", \\"Jupiter\\", \\"Saturn\\")","task_id":"Python/148"},{"entry_point":"sorted_list_sum","public_tests_source":"def check(sorted_list_sum):\\n    # Check some simple cases\\n    assert sorted_list_sum([\\"aa\\", \\"a\\", \\"aaa\\"]) == [\\"aa\\"]\\n    assert sorted_list_sum([\\"ab\\", \\"a\\", \\"aaa\\", \\"cd\\"]) == [\\"ab\\", \\"cd\\"]\\ncheck(sorted_list_sum)\\n","source":"def sorted_list_sum(lst):\\n    lst.sort()\\n    new_lst = []\\n    for i in lst:\\n        if len(i)%2 == 0:\\n            new_lst.append(i)\\n    return new_lst\\n","specification":"Write a function that accepts a list of strings as a parameter,\\ndeletes the strings that have odd lengths from it,\\nand returns the resulted list with a sorted order,\\nThe list is always a list of strings and never an array of numbers,\\nand it may contain duplicates.\\nThe order of the list should be ascending by length of each word, and you\\nshould return the list sorted by that rule.\\nIf two words have the same length, sort the list alphabetically.\\nThe function should return a list of strings in sorted order.\\nYou may assume that all words will have the same length.\\nFor example:\\nassert list_sort([\\"aa\\", \\"a\\", \\"aaa\\"]) => [\\"aa\\"]\\nassert list_sort([\\"ab\\", \\"a\\", \\"aaa\\", \\"cd\\"]) => [\\"ab\\", \\"cd\\"]","task_id":"Python/149"},{"entry_point":"x_or_y","public_tests_source":"def check(x_or_y):\\n    # Check some simple cases\\n    assert x_or_y(7, 34, 12) == 34\\n    assert x_or_y(15, 8, 5) == 5\\ncheck(x_or_y)\\n","source":"def x_or_y(n, x, y):\\n    if n == 1:\\n        return y\\n    for i in range(2, n):\\n        if n % i - 1 == 0:\\n            return y\\n            break\\n    else:\\n        return x\\n","specification":"A simple program which should return the value of x if n is\\na prime number and should return the value of y otherwise.\\nExamples:\\nfor x_or_y(7, 34, 12) == 34\\nfor x_or_y(15, 8, 5) == 5","task_id":"Python/150"},{"entry_point":"double_the_difference","public_tests_source":"def check(double_the_difference):\\n    # Check some simple cases\\n    assert double_the_difference([1,3,2,0]) == 10 , \\"This prints if this assert fails 1 (good for debugging!)\\"\\n    assert double_the_difference([-1,-2,0]) == 0 , \\"This prints if this assert fails 2 (good for debugging!)\\"\\n    assert double_the_difference([9,-2]) == 81 , \\"This prints if this assert fails 3 (good for debugging!)\\"\\n    assert double_the_difference([0]) == 0 , \\"This prints if this assert fails 4 (good for debugging!)\\"\\ncheck(double_the_difference)\\n","source":"def double_the_difference(lst):\\n    return sum([i**2 for i in lst if i > 0 and \\".\\" not in str(i)])\\n","specification":"Given a list of numbers, return the sum of squares of the numbers\\nin the list that are odd. Ignore numbers that are negative or not integers.\\ndouble_the_difference([1, 3, 2, 0]) == 1 + 9 + 0 + 0 = 10\\ndouble_the_difference([-1, -2, 0]) == 0\\ndouble_the_difference([9, -2]) == 81\\ndouble_the_difference([0]) == 0\\nIf the input list is empty, return 0.","task_id":"Python/151"},{"entry_point":"compare","public_tests_source":"def check(compare):\\n    # Check some simple cases\\n    assert compare([1,2,3,4,5,1],[1,2,3,4,2,-2])==[0,0,0,0,3,3], \\"This prints if this assert fails 1 (good for debugging!)\\"\\n    assert compare([0,5,0,0,0,4],[4,1,1,0,0,-2])==[4,4,1,0,0,6]\\ncheck(compare)\\n","source":"def compare(game,guess):\\n    return [abs(x-y)+abs(y-x) for x,y in zip(game,guess)]\\n","specification":"I think we all remember that feeling when the result of some long-awaited\\nevent is finally known. The feelings and thoughts you have at that moment are\\ndefinitely worth noting down and comparing.\\nYour task is to determine if a person correctly guessed the results of a number of matches.\\nYou are given two arrays of scores and guesses of equal length, where each index shows a match.\\nReturn an array of the same length denoting how far off each guess was. If they have guessed correctly,\\nthe value is 0, and if not, the value is the absolute difference between the guess and the score.\\nexample:\\ncompare([1,2,3,4,5,1],[1,2,3,4,2,-2]) -> [0,0,0,0,3,3]\\ncompare([0,5,0,0,0,4],[4,1,1,0,0,-2]) -> [4,4,1,0,0,6]","task_id":"Python/152"},{"entry_point":"Strongest_Extension","public_tests_source":"def check(Strongest_Extension):\\n    # Check some simple cases\\n    assert Strongest_Extension(\'my_class\', [\'AA\', \'Be\', \'CC\']) == \'my_class.AA\'\\ncheck(Strongest_Extension)\\n","source":"def Strongest_Extension(class_name, extensions):\\n    strong = extensions[0]\\n    my_val = len([x for x in extensions[0] if x.isalpha() and x.isupper()]) - len([x for x in extensions[0] if x.isalpha() and x.islower()])\\n    for s in extensions:\\n        val = len([x for x in s if x.isalpha() and x.isupper()]) - len([x for x in s if x.isalpha() and x.islower()])\\n        if val > my_val:\\n            strong = s\\n            my_val = val\\n\\n    ans = class_name + strong\\n    return ans\\n\\n","specification":"You will be given the name of a class (a string) and a list of extensions.\\nThe extensions are to be used to load additional classes to the class. The\\nstrength of the extension is as follows: Let CAP be the number of the uppercase\\nletters in the extension\'s name, and let SM be the number of lowercase letters\\nin the extension\'s name, the strength is given by the fraction CAP - SM.\\nYou should find the strongest extension and return a string in this\\nformat: ClassName.StrongestExtensionName.\\nIf there are two or more extensions with the same strength, you should\\nchoose the one that comes first in the list.\\nFor example, if you are given \\"Slices\\" as the class and a list of the\\nextensions: [\'SErviNGSliCes\', \'Cheese\', \'StuFfed\'] then you should\\nreturn \'Slices.SErviNGSliCes\' since \'SErviNGSliCes\' is the strongest extension\\n(its strength is -1).\\nExample:\\nfor Strongest_Extension(\'my_class\', [\'AA\', \'Be\', \'CC\']) == \'my_class.AA\'","task_id":"Python/153"},{"entry_point":"cycpattern_check","public_tests_source":"def check(cycpattern_check):\\n    # Check some simple cases\\n    #assert True, \\"This prints if this assert fails 1 (good for debugging!)\\"\\n    # Check some edge cases that are easy to work out by hand.\\n    #assert True, \\"This prints if this assert fails 2 (also good for debugging!)\\"\\n    assert  cycpattern_check(\\"abcd\\",\\"abd\\") == False , \\"test #0\\"\\n    assert  cycpattern_check(\\"hello\\",\\"ell\\") == True , \\"test #1\\"\\n    assert  cycpattern_check(\\"whassup\\",\\"psus\\") == False , \\"test #2\\"\\n    assert  cycpattern_check(\\"abab\\",\\"baa\\") == True , \\"test #3\\"\\n    assert  cycpattern_check(\\"efef\\",\\"eeff\\") == False , \\"test #4\\"\\n    assert  cycpattern_check(\\"himenss\\",\\"simen\\") == True , \\"test #5\\"\\ncheck(cycpattern_check)\\n","source":"def cycpattern_check(a , b):\\n    l = len(b)\\n    pat = b + b\\n    for i in range(len(a) - l + 1):\\n        for j in range(len(b) - l + 1):\\n            if a[i:i+l] == pat[j:j+l]:\\n                return True\\n    return False\\n","specification":"You are given 2 words. You need to return True if the second word or any of its rotations is a substring in the first word\\ncycpattern_check(\\"abcd\\",\\"abd\\") => False\\ncycpattern_check(\\"hello\\",\\"ell\\") => True\\ncycpattern_check(\\"whassup\\",\\"psus\\") => False\\ncycpattern_check(\\"abab\\",\\"baa\\") => True\\ncycpattern_check(\\"efef\\",\\"eeff\\") => False\\ncycpattern_check(\\"himenss\\",\\"simen\\") => True","task_id":"Python/154"},{"entry_point":"even_odd_count","public_tests_source":"def check(even_odd_count):\\n    # Check some simple cases\\n    assert even_odd_count(-12) == (1, 1)\\n    assert even_odd_count(123) == (1, 2)\\n    # Check some edge cases that are easy to work out by hand.\\n    assert True\\ncheck(even_odd_count)\\n","source":"def even_odd_count(num):\\n    even_count = 0\\n    odd_count = 0\\n    for i in str(abs(num)):\\n        if int(i)%2==0:\\n            even_count +=1\\n    return (even_count, odd_count)\\n","specification":"Given an integer. return a tuple that has the number of even and odd digits respectively.\\nExample:\\neven_odd_count(-12) ==> (1, 1)\\neven_odd_count(123) ==> (1, 2)","task_id":"Python/155"},{"entry_point":"int_to_mini_roman","public_tests_source":"def check(int_to_mini_roman):\\n    # Check some simple cases\\n    assert int_to_mini_roman(19) == \'xix\'\\n    assert int_to_mini_roman(152) == \'clii\'\\n    assert int_to_mini_roman(426) == \'cdxxvi\'\\ncheck(int_to_mini_roman)\\n","source":"def int_to_mini_roman(number):\\n    num = [1, 4, 5, 9, 10, 40, 50, 90,  \\n           100, 400, 500, 900, 1000] \\n    sym = [\\"I\\", \\"IV\\", \\"V\\", \\"IX\\", \\"X\\", \\"XL\\",  \\n           \\"L\\", \\"XC\\", \\"C\\", \\"CD\\", \\"D\\", \\"CM\\", \\"M\\"] \\n    i = 12\\n    res = \'\'\\n    while number: \\n        div = number // num[i] \\n        number %= num[i] \\n        while div: \\n            res += sym[i]\\n        i -= 1\\n    return res.lower()\\n","specification":"Given a positive integer, obtain its roman numeral equivalent as a string,\\nand return it in lowercase.\\nRestrictions: 1 <= num <= 1000\\nExamples:\\n>>> int_to_mini_roman(19) == \'xix\'\\n>>> int_to_mini_roman(152) == \'clii\'\\n>>> int_to_mini_roman(426) == \'cdxxvi\'","task_id":"Python/156"},{"entry_point":"right_angle_triangle","public_tests_source":"def check(right_angle_triangle):\\n    # Check some simple cases\\n    assert right_angle_triangle(3, 4, 5) == True, \\"This prints if this assert fails 1 (good for debugging!)\\"\\n    assert right_angle_triangle(1, 2, 3) == False\\ncheck(right_angle_triangle)\\n","source":"def right_angle_triangle(a, b, c):\\n    return c*c == a*a + b*b\\n","specification":"Given the lengths of the three sides of a triangle. Return True if the three\\nsides form a right-angled triangle, False otherwise.\\nA right-angled triangle is a triangle in which one angle is right angle or\\n90 degree.\\nExample:\\nright_angle_triangle(3, 4, 5) == True\\nright_angle_triangle(1, 2, 3) == False","task_id":"Python/157"},{"entry_point":"find_max","public_tests_source":"def check(find_max):\\n    # Check some simple cases\\n    assert (find_max([\\"name\\", \\"of\\", \\"string\\"]) == \\"string\\"), \\"t1\\"\\n    assert (find_max([\\"name\\", \\"enam\\", \\"game\\"]) == \\"enam\\"), \'t2\'\\n    assert (find_max([\\"aaaaaaa\\", \\"bb\\", \\"cc\\"]) == \\"aaaaaaa\\"), \'t3\'\\ncheck(find_max)\\n","source":"def find_max(words):\\n    return sorted(words)[0]\\n","specification":"Write a function that accepts a list of strings.\\nThe list contains different words. Return the word with maximum number\\nof unique characters. If multiple strings have maximum number of unique\\ncharacters, return the one which comes first in lexicographical order.\\nfind_max([\\"name\\", \\"of\\", \\"string\\"]) == \\"string\\"\\nfind_max([\\"name\\", \\"enam\\", \\"game\\"]) == \\"enam\\"\\nfind_max([\\"aaaaaaa\\", \\"bb\\" ,\\"cc\\"]) == \\"\\"aaaaaaa\\"","task_id":"Python/158"},{"entry_point":"eat","public_tests_source":"def check(eat):\\n    # Check some simple cases\\n    assert True, \\"This prints if this assert fails 1 (good for debugging!)\\"\\n    assert eat(5, 6, 10) == [11, 4], \\"Error\\"\\n    assert eat(4, 8, 9) == [12, 1], \\"Error\\"\\n    assert eat(1, 10, 10) == [11, 0], \\"Error\\"\\n    assert eat(2, 11, 5) == [7, 0], \\"Error\\"\\n    # Check some edge cases that are easy to work out by hand.\\n    assert True, \\"This prints if this assert fails 2 (also good for debugging!)\\"\\ncheck(eat)\\n","source":"def eat(number, need, remaining):\\n    if(need <= remaining):\\n        return [ number + need , number + remaining-need ]\\n    else:\\n        return [ number + need + remaining , 0]\\n","specification":"You\'re a hungry rabbit, and you already have eaten a certain number of carrots,\\nbut now you need to eat more carrots to complete the day\'s meals.\\nyou should return an array of [ total number of eaten carrots after your meals,\\nthe number of carrots left after your meals ]\\nif there are not enough remaining carrots, you will eat all remaining carrots, but will still be hungry.\\nExample:\\n* eat(5, 6, 10) -> [11, 4]\\n* eat(4, 8, 9) -> [12, 1]\\n* eat(1, 10, 10) -> [11, 0]\\n* eat(2, 11, 5) -> [7, 0]\\nVariables:\\n@number : integer\\nthe number of carrots that you have eaten.\\n@need : integer\\nthe number of carrots that you need to eat.\\n@remaining : integer\\nthe number of remaining carrots thet exist in stock\\nConstrain:\\n* 0 <= number <= 1000\\n* 0 <= need <= 1000\\n* 0 <= remaining <= 1000\\nHave fun :)","task_id":"Python/159"},{"entry_point":"do_algebra","public_tests_source":"","source":"def do_algebra(operator, operand):\\n    expression = str(operand[0])\\n    for oprt, oprn in zip(operator, operand[1:]):\\n        expression+=expression + oprt + str(oprn)\\n    return eval(expression)\\n","specification":"Given two lists operator, and operand. The first list has basic algebra operations, and\\nthe second list is a list of integers. Use the two given lists to build the algebric\\nexpression and return the evaluation of this expression.\\nThe basic algebra operations:\\nAddition ( + )\\nSubtraction ( - )\\nMultiplication ( * )\\nFloor division ( // )\\nExponentiation ( ** )\\nExample:\\noperator[\'+\', \'*\', \'-\']\\narray = [2, 3, 4, 5]\\nresult = 2 + 3 * 4 - 5\\n=> result = 9\\nNote:\\nThe length of operator list is equal to the length of operand list minus one.\\nOperand is a list of of non-negative integers.\\nOperator list has at least one operator, and operand list has at least two operands.","task_id":"Python/160"},{"entry_point":"solve","public_tests_source":"def check(solve):\\n    # Check some simple cases\\n    assert solve(\\"1234\\") == \\"4321\\"\\n    assert solve(\\"ab\\") == \\"AB\\"\\n    assert solve(\\"#a@C\\") == \\"#A@c\\"\\n    # Don\'t remove this line:\\ncheck(solve)\\n","source":"def solve(s):\\n    flg = 0\\n    idx = 0\\n    new_str = list(s)\\n    for i in s:\\n        if i.isalpha():\\n            new_str[idx] = i\\n            flg = 1\\n        idx += 1\\n    s = \\"\\"\\n    for i in new_str:\\n        s += i\\n    if flg == 0:\\n        return s[len(s)::-1]\\n    return s\\n","specification":"You are given a string s.\\nif s[i] is a letter, reverse its case from lower to upper or vise versa,\\notherwise keep it as it is.\\nIf the string contains no letters, reverse the string.\\nThe function should return the resulted string.\\nExamples\\nsolve(\\"1234\\") = \\"4321\\"\\nsolve(\\"ab\\") = \\"AB\\"\\nsolve(\\"#a@C\\") = \\"#A@c\\"","task_id":"Python/161"},{"entry_point":"string_to_md5","public_tests_source":"def check(string_to_md5):\\n    # Check some simple cases\\n    assert string_to_md5(\'Hello world\') == \'3e25960a79dbc69b674cd4ec67a72c62\'\\n    # Check some edge cases that are easy to work out by hand.\\n    assert True\\ncheck(string_to_md5)\\n","source":"def string_to_md5(text):\\n    import hashlib\\n    return hashlib.md5(\'text\').hexdigest() if text else None\\n","specification":"Given a string \'text\', return its md5 hash equivalent string.\\nIf \'text\' is an empty string, return None.\\n>>> string_to_md5(\'Hello world\') == \'3e25960a79dbc69b674cd4ec67a72c62\'","task_id":"Python/162"},{"entry_point":"generate_integers","public_tests_source":"def check(generate_integers):\\n    # Check some simple cases\\n    assert generate_integers(2, 10) == [2, 4, 6, 8], \\"Test 1\\"\\n    assert generate_integers(10, 2) == [2, 4, 6, 8], \\"Test 2\\"\\n    assert generate_integers(132, 2) == [2, 4, 6, 8], \\"Test 3\\"\\n    assert generate_integers(17,89) == [], \\"Test 4\\"\\n    # Check some edge cases that are easy to work out by hand.\\n    assert True, \\"This prints if this assert fails 2 (also good for debugging!)\\"\\ncheck(generate_integers)\\n","source":"def generate_integers(a, b):\\n    lower = max(2, min(a, b))\\n    upper = min(8, max(a, b))\\n\\n    return [i for i in range(lower, upper) if i % 2 == 0]\\n","specification":"Given two positive integers a and b, return the even digits between a\\nand b, in ascending order.\\nFor example:\\ngenerate_integers(2, 8) => [2, 4, 6, 8]\\ngenerate_integers(8, 2) => [2, 4, 6, 8]\\ngenerate_integers(10, 14) => []","task_id":"Python/163"}]')
MANIFEST_SHA256 = 'aa12497c58ef8639dacf41c42f972366e9f916924200336f843a3d14d3ac4400'
assert len(TASKS) == 164
print('Public tasks:', len(TASKS))


In [ ]:
import importlib.metadata
import os
from pathlib import Path
import random
import time
from datetime import datetime, timezone
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL = "Qwen/Qwen3-0.6B"
REVISION = "c1899de289a04d12100db370d81485cdf75e47ca"
SEED = 7
MAX_NEW_TOKENS = 1024
SYSTEM = 'Repair the buggy Python function to satisfy its specification and examples. Return ONLY the complete corrected Python source, including necessary standard-library imports. Preserve the supplied entry-point name and signature. Do not include Markdown, tests, or explanations. You cannot execute code or use tools. You get one response.'
random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cuda.matmul.allow_tf32 = False
OUTPUT = Path("repairlab-baseline-" + datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ"))
OUTPUT.mkdir(exist_ok=False)
(OUTPUT / "raw").mkdir()
dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
metadata = {
    "protocol": 'humanevalfix-docs-examples-single-response-v1', "split": "external-public-test", "model": MODEL,
    "model_revision": REVISION, "adapter": None, "seed": SEED,
    "manifest_sha256": MANIFEST_SHA256, "system_message": SYSTEM,
    "decoding": {"do_sample": False, "num_beams": 1, "num_return_sequences": 1,
                 "max_new_tokens": MAX_NEW_TOKENS, "enable_thinking": False,
                 "max_input_tokens": 3072},
    "source_extraction": "strip-whitespace-and-one-enclosing-code-fence-v1",
    "gpu": torch.cuda.get_device_name(0), "dtype": str(dtype),
    "packages": {p: importlib.metadata.version(p) for p in
                 ("torch", "transformers", "tokenizers", "huggingface-hub", "safetensors")},
    "created_at": datetime.now(timezone.utc).isoformat(),
}
(OUTPUT / "model-config.json").write_text(json.dumps(metadata, indent=2))
tokenizer = AutoTokenizer.from_pretrained(MODEL, revision=REVISION, trust_remote_code=False)
model = AutoModelForCausalLM.from_pretrained(
    MODEL, revision=REVISION, torch_dtype=dtype, trust_remote_code=False
).to("cuda:0").eval()
print("Loaded", MODEL, REVISION, "Output:", OUTPUT)


In [ ]:
# No exec/eval/import of model output. All patches remain text until local grading.
def extract_source(raw):
    text = raw.strip()
    lines = text.splitlines()
    if len(lines) >= 3 and lines[0] in ("```python", "```") and lines[-1] == "```":
        return "\n".join(lines[1:-1]) + "\n"
    return text + "\n"

started = time.monotonic()
with (OUTPUT / "predictions.jsonl").open("w") as predictions:
    for index, task in enumerate(TASKS):
        begin = time.monotonic()
        row = {"task_id": task["task_id"]}
        try:
            messages = [{"role": "system", "content": SYSTEM},
                        {"role": "user", "content": json.dumps(task, ensure_ascii=False)}]
            prompt = tokenizer.apply_chat_template(
                messages, tokenize=False, add_generation_prompt=True, enable_thinking=False
            )
            inputs = tokenizer(prompt, return_tensors="pt").to("cuda:0")
            if inputs.input_ids.shape[1] > 3072:
                raise ValueError("Input exceeds the fixed 3072-token budget")
            with torch.inference_mode():
                generated = model.generate(
                    **inputs, do_sample=False, num_beams=1, num_return_sequences=1,
                    max_new_tokens=MAX_NEW_TOKENS, pad_token_id=tokenizer.eos_token_id
                )
            completion = generated[0, inputs.input_ids.shape[1]:]
            raw = tokenizer.decode(completion, skip_special_tokens=True)
            filename = task["task_id"].replace("/", "-") + ".txt"
            (OUTPUT / "raw" / filename).write_text(raw)
            row.update(source=extract_source(raw), raw_output=raw,
                       input_tokens=int(inputs.input_ids.shape[1]),
                       output_tokens=int(completion.shape[0]))
        except Exception as exc:
            row["generation_error"] = type(exc).__name__ + ": " + str(exc)[:1000]
        row["elapsed_seconds"] = time.monotonic() - begin
        predictions.write(json.dumps(row, ensure_ascii=False) + "\n")
        predictions.flush()
        print(index + 1, "/", len(TASKS), task["task_id"],
              "ERROR" if "generation_error" in row else "saved; not yet graded", flush=True)
        if row.get("generation_error", "").startswith("OutOfMemoryError"):
            print("Stopping after GPU memory failure; do not retry unchanged settings.")
            break
(OUTPUT / "runtime.json").write_text(json.dumps({
    "elapsed_seconds": time.monotonic() - started,
    "peak_gpu_bytes": torch.cuda.max_memory_allocated(0),
    "note": "Inference only. No repair execution, correctness scores, or weight updates."
}, indent=2))


In [ ]:
import shutil
archive = shutil.make_archive(str(OUTPUT), "zip", OUTPUT)
print("Download:", archive)
try:
    from google.colab import files
    files.download(archive)
except ImportError:
    from IPython.display import FileLink, display
    display(FileLink(archive))


## Grade locally

Download and extract the archive. From the local `repairlab` directory, with
Docker running:

```sh
python3 -B -m repairlab external-score \
  --predictions /absolute/path/to/extracted/predictions.jsonl \
  --metadata /absolute/path/to/extracted/model-config.json \
  --output artifacts/baseline-001
```

All 164 tasks remain in the denominator, including missing, malformed, and
failed generations. Infrastructure failures stop scoring and must be diagnosed.
Scores on these development tasks guide our next experiment; they are not a
final test result. Do not train on them and later call their scores held-out.


This is **RepairLab's HumanEvalFix Python adaptation**, using docstrings AND
public examples, complete-source responses, and the original assertion tests.
Prompting, extraction, helper imports, and execution limits differ from the
authors' harness. These scores are not a reproduction of published leaderboard
results. The local external assert runner is not a reward-hacking-resistant RL
oracle. Do not train on these public test tasks.
